# Cross-hospital generalization of chest X-ray classifiers: one master notebook

**Sites:** A = NIH ChestX-ray14, B = CheXpert, C = VinBigData (VinDr-CXR images, radiologist labels, no Edema).
**Architectures:** DenseNet121, ConvNeXt-T, ViT-S. **Training sets:** A, B, C, A+B, A+C, B+C. **Seeds:** 3.
**DG methods on pooled sets:** ERM (natural), ERM (balanced), DANN, CORAL, GroupDRO.

**How to use this notebook**

1. Edit only the block marked `EDIT ONLY THIS BLOCK` in the first code cell.
2. Run the cells from top to bottom. Every training cell **skips runs that are already finished** and resumes interrupted ones, so a full run on your machine only retrains what is missing.
3. DenseNet121 is **not retrained here**. Its predictions come from your earlier runs (`fair_ens` logits). ConvNeXt-T ERM (natural) is loaded the same way.
4. `FAST = True` makes everything about 7x quicker but the bootstrap p-values are then too coarse for Holm correction. **Use `FAST = False` for every number that goes into the paper.**

**Sections**

| Part | Content |
|---|---|
| A | Setup, constants, metrics, data, dataset tables |
| B | Training (ViT-S all runs, ConvNeXt-T DG runs), skip if finished |
| C | Load all predictions, coverage check |
| D | Main results: transfer matrices, AUROC and AUPRC |
| E | Statistics: paired bootstrap, gaps, asymmetry, architecture differences |
| F | What explains the gap: view/case mix, prevalence shift |
| G | Calibration, label-free prior-shift correction, k-label recalibration |
| H | Domain generalization methods vs ERM |
| I | Ensembles and diversity |
| J | Representation, site probe, Grad-CAM |
| K | Cost, summary, export, methods text, limitations |

## Project status when this notebook was built

**Done (verified in your notebooks and outputs)**

| Area | Status |
|---|---|
| Data: NIH (official test list), CheXpert (patient split), VinBigData (2-of-3 radiologist rule), 224 px caches | done |
| DenseNet121: ERM natural on A, B, C, A+B, A+C, B+C; ERM balanced, DANN, CORAL on all three pooled sets, 3 seeds | done (GroupDRO only on A+B, without VinBigData predictions) |
| ConvNeXt-T: ERM natural on six sets; ERM balanced, DANN, CORAL, GroupDRO on three pooled sets, 3 seeds | done (`convnext.ipynb`, 0 errors) |
| ViT-S: ERM natural on six sets; four DG methods on three pooled sets, 3 seeds | done (`DASDA.ipynb`, 0 errors) |
| Transfer matrices, gaps, asymmetry with paired patient bootstrap, architecture differences | done, rerun here with NB = 2000 |
| Calibration, k-label recalibration, ensembles, site probe, t-SNE, Grad-CAM, efficiency | done, regenerated here for all three architectures |
| Size-matched control and PA/AP-stratified gaps (DenseNet121) | done in `che44-nih77.ipynb`; its last bootstrap cell stopped at 46% in the saved copy |
| Stage 2 ensemble test (verdict REVERSED), Stage 3 equal-budget tuning | run, see the cautions below |

**Open or needs care**

1. **Holm correction needs more resamples.** With 300 resamples the smallest attainable p is 1/300, so Holm over about 45 comparisons can never reach 0.05. "No significant architecture difference" from a 300-resample run is therefore a resolution artifact. This notebook uses 2000.
2. **DG methods use balanced sampling, ERM (natural) does not.** The fair baseline for a DG method is ERM (balanced). Part H reports both.
3. **GroupDRO can collapse onto one hospital** (`q = [1, 0]` in your ConvNeXt B+C log). Part H tabulates the final weights. Report them.
4. **Stage 3 (`REFER2`) CORAL trials never trained** (validation AUROC about 0.5 and constant loss in every trial, while your CORAL in the main runs trains normally at about 0.79). Do not cite those trials as evidence about CORAL. Rerun them after checking for skipped optimizer steps (non-finite gradients) in that notebook.
5. DenseNet121 GroupDRO on the pooled sets with VinBigData predictions is missing.
6. No MIMIC-CXR or PadChest, and no 512 px check.
7. Still to do outside this notebook: label-harmonization write-up, code release, supervisor review, venue choice.

In [ ]:
import os, gc, json, math, time, warnings, glob, shutil, platform
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.manifold import TSNE
from itertools import combinations
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **k: x
warnings.filterwarnings("ignore")

# =========================== EDIT ONLY THIS BLOCK ===========================
OUT   = os.environ.get("CXR_OUT", r"G:\runs")                      # root folder of everything you trained so far
PATHS = dict(nih_split=r"G:\nih_split.csv", nih_u8=r"G:\nih_224.u8", cx_csv=r"G:\chexpert_final_clean.csv",
             cx_trainval_u8=r"G:\cx_trainval_224.u8", cx_test_npy=r"G:\cx_test_224.npy", vb_dir=os.path.join(OUT, "vinbig"))
EPOCHS, SEEDS, BS = 7, [0, 1, 2], 64
FAST = False        # True = 300 bootstrap resamples (quick look only). False = 2000, needed for Holm-corrected p-values
# ============================================================================
NB      = 300 if FAST else 2000       # bootstrap resamples, main comparisons
NB_ENS  = 300 if FAST else 1000       # bootstrap resamples, ensembles
NB_VIEW = 300 if FAST else 500        # bootstrap resamples, view-stratified analysis
MASTER = os.path.join(OUT, "master"); TABD = os.path.join(MASTER, "tables"); FIG = os.path.join(MASTER, "figures")
for d in (MASTER, TABD, FIG): os.makedirs(d, exist_ok=True)

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"], "mathtext.fontset": "stix",
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "lines.linewidth": 1.0, "grid.linewidth": 0.4, "figure.dpi": 150, "pdf.fonttype": 42})
COL1, COL2 = 3.5, 7.16                                              # single / double column width in inches
def save_fig(fig, name):
    fig.savefig(os.path.join(FIG, name + ".pdf"), bbox_inches="tight"); fig.savefig(os.path.join(FIG, name + ".png"), dpi=600, bbox_inches="tight"); plt.show()
def save_tab(df, name, index=True):
    df.to_csv(os.path.join(TABD, name + ".csv"), index=index)
    try: open(os.path.join(TABD, name + ".tex"), "w").write(df.to_latex(index=index, escape=True))
    except Exception: pass
    return df
print("notebook root:", MASTER, "| bootstrap NB =", NB)

In [ ]:
import torch, timm, torch.nn as nn, torch.nn.functional as F
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| timm", timm.__version__, "| device:", torch.cuda.get_device_name(0) if dev == "cuda" else "CPU (training will be very slow)")

In [ ]:
LABELS6 = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
FIND5   = ["Cardiomegaly", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]; L = 6
SITE = {"A": "NIH", "B": "CheXpert", "C": "VinBigData"}
TS = ["A", "B", "C", "A+B", "A+C", "B+C"]                            # the six training sets
POOLED = ["A+B", "A+C", "B+C"]; UNSEEN = {"A+B": "C", "A+C": "B", "B+C": "A"}
SETS_ORDER = ["A+B", "A+C", "B+C"]; SEEN_T = {"A+B": ["A", "B"], "A+C": ["A"], "B+C": ["B"]}   # seen hospitals that have a clean held-out test set
NAT = "ERM (natural)"; METHS = [NAT, "ERM (balanced)", "DANN", "CORAL", "GroupDRO"]
TAG = {NAT: "", "ERM (balanced)": "erm_bal", "DANN": "dann", "CORAL": "coral", "GroupDRO": "gdro"}
A3 = ["DenseNet121", "ConvNeXt-T", "ViT-S"]
AC  = {"DenseNet121": "0.2", "ConvNeXt-T": "0.55", "ViT-S": "0.85"}   # grey levels, print friendly
ALS = {"DenseNet121": "-", "ConvNeXt-T": "--", "ViT-S": ":"}; MKR = {"DenseNet121": "o", "ConvNeXt-T": "^", "ViT-S": "s"}
MC = {NAT: "0.12", "ERM (balanced)": "0.38", "DANN": "0.58", "CORAL": "0.75", "GroupDRO": "0.9"}
HT = {NAT: "", "ERM (balanced)": "///", "DANN": "\\\\\\", "CORAL": "xxx", "GroupDRO": "..."}
ARCH_CFG = {
  "ViT-S":       dict(timm="vit_small_patch16_224.augreg_in21k_ft_in1k", lr=5e-5, wd=0.05, dir=os.path.join(OUT, "vit"),        prefix="vit", drop_path=0.1),
  "ConvNeXt-T":  dict(timm="convnext_tiny.fb_in1k",                       lr=1e-4, wd=1e-4, dir=os.path.join(OUT, "convnext_dg"), prefix="cnx", drop_path=0.0),
  "DenseNet121": dict(timm="densenet121",                                 lr=1e-4, wd=1e-4, dir=None,                            prefix=None,  drop_path=0.0)}
def stem(a, ts, me, s):                                              # file stem of a model trained by this notebook (ViT-S, ConvNeXt-T)
    c = ARCH_CFG[a]; return os.path.join(c["dir"], f"{c['prefix']}_{ts.replace('+', 'p')}_seed{s}") + (("_" + TAG[me]) if TAG[me] else "")
FE = os.path.join(OUT, "fair_ens")                                   # DenseNet121 and ConvNeXt-T ERM (natural) logits from your earlier runs

In [ ]:
def fauc(y, s):
    y = np.asarray(y).astype(bool); r = rankdata(s); n1 = y.sum(); n0 = len(y) - n1
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
def lab_auc(Y, P, valid):                                            # per-label AUROC, NaN where the label is absent or too rare
    return np.array([fauc(Y[:, j], P[:, j]) if valid[j] and 5 <= Y[:, j].sum() <= len(Y) - 5 else np.nan for j in range(L)])
def mauc(Y, P, valid): return float(np.nanmean(lab_auc(Y, P, valid)))
def mauprc(Y, P, valid): return float(np.mean([average_precision_score(Y[:, j], P[:, j]) for j in range(L) if valid[j] and Y[:, j].sum() >= 5]))
sig = lambda z: 1 / (1 + np.exp(-z))
def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
GRID = np.linspace(0.02, 0.95, 94)
def best_thr(y, p):                                                  # threshold that maximises F1 on (y, p)
    pr = p[:, None] >= GRID[None, :]; y1 = (y == 1)[:, None]; tp = (pr & y1).sum(0); fp = pr.sum(0) - tp; fn = y1.sum() - tp
    return GRID[np.argmax(2 * tp / np.maximum(2 * tp + fp + fn, 1))]
def f1_at(y, p, t):
    pr = p >= t; y1 = y == 1; tp = (pr & y1).sum(); d = 2 * tp + (pr & ~y1).sum() + (~pr & y1).sum(); return 2 * tp / d if d else 0.0
def ece_of(y, p, nb=10):
    bi = np.clip((p * nb).astype(int), 0, nb - 1); e = 0.0
    for k in range(nb):
        mk = bi == k
        if mk.any(): e += mk.mean() * abs(p[mk].mean() - y[mk].mean())
    return e
def reliab(y, p, nb=10):
    bi = np.clip((p * nb).astype(int), 0, nb - 1); xs_, ys_ = [], []
    for k in range(nb):
        mk = bi == k
        if mk.sum() > 30: xs_.append(p[mk].mean()); ys_.append(y[mk].mean())
    return xs_, ys_
def holm(p):                                                         # Holm step-down adjusted p-values
    p = np.asarray(p, float); o = np.argsort(p); out = np.empty(len(p)); run = 0.0
    for r, i in enumerate(o): run = max(run, min(1.0, (len(p) - r) * p[i])); out[i] = run
    return out
def boot_p(d, nb): return min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()) + 1.0 / nb)    # two-sided bootstrap p; floor is 1/nb, so nb must be large for Holm
pci = lambda a: np.percentile(a, [2.5, 97.5], axis=0)
def groups(p): return list(pd.Series(np.arange(len(p))).groupby(np.asarray(p)).apply(np.array).values)
def resample(T, rng):                                                # patient-level bootstrap indices for test site T (image-level for C: no patient id)
    if GRP[T] is None: n = len(YT[T]); return rng.integers(0, n, n)
    g = GRP[T]; return np.concatenate([g[k] for k in rng.integers(0, len(g), len(g))])
dname = lambda ts, T: f"{ts}→{T}"

In [ ]:
# ---------------- pre-flight: are all the inputs where this notebook expects them? ----------------
need = {"NIH split csv": PATHS["nih_split"], "NIH 224 cache": PATHS["nih_u8"], "CheXpert csv": PATHS["cx_csv"], "CheXpert train/val cache": PATHS["cx_trainval_u8"],
        "CheXpert test cache": PATHS["cx_test_npy"], "VinBigData labels": os.path.join(PATHS["vb_dir"], "vinbig_image_labels.csv"), "VinBigData cache": os.path.join(PATHS["vb_dir"], "vb_train_224.u8"),
        "DenseNet/ConvNeXt logits folder (fair_ens)": FE}
opt = {"ViT-S folder": ARCH_CFG["ViT-S"]["dir"], "ConvNeXt DG folder": ARCH_CFG["ConvNeXt-T"]["dir"], "NIH-trained DenseNet checkpoints": os.path.join(OUT, "nih_train"),
       "ConvNeXt single-source checkpoints (part34)": os.path.join(OUT, "part34"), "DenseNet A+B checkpoints (dg)": os.path.join(OUT, "dg"), "ConvNeXt A+B checkpoints (c_source)": os.path.join(OUT, "c_source")}
for nm, pth in need.items(): print(("OK      " if os.path.exists(pth) else "MISSING ") + nm + "  ->  " + pth)
print("--- needed only for training / Grad-CAM / recalibration (skipped with a message when absent) ---")
for nm, pth in opt.items(): print(("OK      " if os.path.exists(pth) else "missing ") + nm + "  ->  " + pth)
assert all(os.path.exists(v) for v in need.values()), "fix the MISSING paths in the first code cell before continuing"

In [ ]:
# ---------------- data: same files and splits as your earlier notebooks ----------------
nih = pd.read_csv(PATHS["nih_split"]); off = nih.official.values == "test"; te_i = np.where(off)[0]
tr_i = np.where(nih.split_rev.values == "train")[0]; va_i = np.where(nih.split_rev.values == "val")[0]
NIH_A = np.memmap(PATHS["nih_u8"], dtype=np.uint8, mode="r", shape=(len(nih), 224, 224)); Yn = nih[LABELS6].values.astype(int)

cx = pd.read_csv(PATHS["cx_csv"]); cxt = cx[cx.split == "test"].reset_index(drop=True)
cxm = cx[cx.split.isin(["train", "val"])].reset_index(drop=True)
cx_tr = np.where(cxm.split.values == "train")[0]; cx_va = np.where(cxm.split.values == "val")[0]
CX_A = np.memmap(PATHS["cx_trainval_u8"], dtype=np.uint8, mode="r", shape=(len(cxm), 224, 224)); CX_T = np.load(PATHS["cx_test_npy"])
Ycm = cxm[LABELS6].values.astype(int); Yc = cxt[LABELS6].values.astype(int)

vb = pd.read_csv(os.path.join(PATHS["vb_dir"], "vinbig_image_labels.csv")); NVB = len(vb)
VB_A = np.memmap(os.path.join(PATHS["vb_dir"], "vb_train_224.u8"), dtype=np.uint8, mode="r", shape=(NVB, 224, 224))
Yvb = np.zeros((NVB, L), int)
for f in FIND5: Yvb[:, LABELS6.index(f)] = vb[f].values
VALID_C = np.array([l != "Edema" for l in LABELS6])                  # VinBigData has no Edema: masked in the loss and in every metric
sp = os.path.join(OUT, "fair_ens", "vinbig_split.csv")               # same C split as your full11 runs
if os.path.exists(sp):
    s_ = pd.read_csv(sp); vb_va = np.where(s_.split.values == "val")[0]; vb_tr = np.where(s_.split.values == "train")[0]
else:
    perm = np.random.default_rng(0).permutation(NVB); vb_va = np.sort(perm[:NVB // 5]); vb_tr = np.sort(perm[NVB // 5:])

SRC = {"A": dict(arr=NIH_A, Y=Yn,  tr=tr_i,  va=va_i,  valid=np.ones(L, bool)),
       "B": dict(arr=CX_A,  Y=Ycm, tr=cx_tr, va=cx_va, valid=np.ones(L, bool)),
       "C": dict(arr=VB_A,  Y=Yvb, tr=vb_tr, va=vb_va, valid=VALID_C)}
TEST = {"A": (NIH_A, te_i), "B": (CX_T, np.arange(len(CX_T))), "C": (VB_A, np.arange(NVB))}   # C is a test site only when it was not a training source
YT = {"A": Yn[te_i], "B": Yc, "C": Yvb}; VALID = {"A": np.ones(L, bool), "B": np.ones(L, bool), "C": VALID_C}
GRP = {"A": groups(nih.patient_id.values[off]), "B": groups(cxt.patient_id.values), "C": None}

# view position (PA/AP): used for the case-mix analysis. CheXpert coding in your mirror: 0 = AP, 1 = PA (checked in che44-nih77)
AP_CODE, PA_CODE = 0, 1
try:
    VIEW = {"A": nih["view"].values[off],
            "B": np.where(cxt["AP/PA"].values == AP_CODE, "AP", np.where(cxt["AP/PA"].values == PA_CODE, "PA", "other"))}
except Exception as e:
    VIEW = None; print("view columns not found, the view analysis will be skipped:", e)
print({k: (len(v["tr"]), len(v["va"])) for k, v in SRC.items()}, "| test:", len(te_i), len(CX_T), "| C images:", NVB)

In [ ]:
# ---------------- Table 1 and Fig. 1: what the three sites look like ----------------
rows = [dict(site="NIH ChestX-ray14 (A)",  train=len(SRC["A"]["tr"]), val=len(SRC["A"]["va"]), test=len(YT["A"])),
        dict(site="CheXpert (B)",           train=len(SRC["B"]["tr"]), val=len(SRC["B"]["va"]), test=len(YT["B"])),
        dict(site="VinBigData (C)",         train=len(SRC["C"]["tr"]), val=len(SRC["C"]["va"]), test=0)]
T1 = pd.DataFrame(rows); T1["note"] = ["official NIH test list", "patient-level split", "all 15,000 images are used as a test set only when C is not a training source"]
display(T1); save_tab(T1, "table1_dataset_sizes", index=False)

rows = []
for T in "ABC":
    for j, l in enumerate(LABELS6):
        rows.append(dict(site=f"{SITE[T]} ({T})", finding=l, prevalence=float(YT[T][:, j].mean()) if VALID[T][j] else np.nan, positives=int(YT[T][:, j].sum()) if VALID[T][j] else 0, n=len(YT[T])))
PV = pd.DataFrame(rows); PVp = PV.pivot(index="finding", columns="site", values="prevalence").reindex(LABELS6); display(PVp.round(3)); save_tab(PVp.round(4), "table2_prevalence_by_site")
if VIEW is not None:
    vm = pd.DataFrame({f"{SITE[T]} ({T})": pd.Series(VIEW[T]).value_counts(normalize=True) for T in "AB"}).round(3); display(vm); save_tab(vm, "table3_view_mix")

fig, ax = plt.subplots(figsize=(COL2, 2.7)); w = .26
for i, T in enumerate("ABC"):
    g = PV[PV.site == f"{SITE[T]} ({T})"].set_index("finding").reindex(LABELS6).prevalence
    ax.bar(np.arange(L) + (i - 1) * w, g.values, w, color=["0.25", "0.55", "0.85"][i], edgecolor="k", lw=.4, label=f"{SITE[T]} ({T})")
ax.set_xticks(range(L)); ax.set_xticklabels(LABELS6, rotation=25, ha="right"); ax.set(ylabel="Prevalence in the test set", title="Disease prevalence differs strongly between sites (VinBigData has no Edema)")
ax.legend(frameon=False); ax.grid(True, axis="y", alpha=.3); fig.tight_layout(); save_fig(fig, "fig01_prevalence_by_site")

# Part B: models and training (skips finished runs)

* **ViT-S:** ERM (natural) on all six training sets, plus ERM (balanced), DANN, CORAL, GroupDRO on the three pooled sets.
* **ConvNeXt-T:** the four DG methods on the three pooled sets. ConvNeXt-T ERM (natural) and all DenseNet121 models come from your earlier runs (`fair_ens` logits) and are not retrained.
* **Recipes** follow each architecture's standard fine-tuning setup and are reported as such in the paper: DenseNet121 and ConvNeXt-T use AdamW, LR 1e-4, weight decay 1e-4, cosine schedule, no warmup, no gradient clipping. ViT-S uses LR 5e-5, weight decay 0.05, 300 warmup steps, gradient clipping 1.0 and drop-path 0.1.
* **DG settings** are identical to your DenseNet runs: DANN weight 0.1 with the lambda ramp, CORAL weight 1.0, GroupDRO eta 0.01. Model selection is always the mean source-validation AUROC over the training hospitals.
* **Known property of GroupDRO here:** it uses the raw group loss as its hardness measure. If one hospital has a higher label prevalence its loss is always larger, and the weights can collapse onto it (`q = [1, 0]` in your ConvNeXt B+C log). The final `q` weights are tabulated in Part H and must be reported.

In [ ]:
IMN_M, IMN_S = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
try:
    _c = timm.create_model(ARCH_CFG["ViT-S"]["timm"], pretrained=False, num_classes=L).pretrained_cfg; _vm, _vs = list(_c["mean"]), list(_c["std"])
except Exception:
    _vm, _vs = [0.5] * 3, [0.5] * 3                                                  # augreg ViT weights use 0.5 / 0.5
NORM = {a: (torch.tensor(m_, device=dev).view(1, 3, 1, 1), torch.tensor(s_, device=dev).view(1, 3, 1, 1))
        for a, (m_, s_) in {"DenseNet121": (IMN_M, IMN_S), "ConvNeXt-T": (IMN_M, IMN_S), "ViT-S": (_vm, _vs)}.items()}
CUR = {"a": "ViT-S"}
def use_arch(a):
    CUR.clear(); CUR.update(ARCH_CFG[a]); CUR["a"] = a
    if ARCH_CFG[a]["dir"]: os.makedirs(ARCH_CFG[a]["dir"], exist_ok=True)
def make_model(pretrained=True, a=None):
    c = ARCH_CFG[a or CUR["a"]]; kw = dict(drop_path_rate=c["drop_path"]) if c["drop_path"] > 0 else {}
    return timm.create_model(c["timm"], pretrained=pretrained, num_classes=L, **kw).to(dev)
def gpu_batch(A, idx, flip=False, a=None):
    mu, sd = NORM[a or CUR["a"]]
    x = torch.from_numpy(np.ascontiguousarray(A[np.asarray(idx)])).to(dev).float().div(255).unsqueeze(1)
    if flip:
        f = torch.rand(len(idx), device=dev) < 0.5; x = torch.where(f.view(-1, 1, 1, 1), x.flip(-1), x)
    return (x.repeat(1, 3, 1, 1) - mu) / sd
@torch.inference_mode()
def predict(m, A, idx, bs=128, a=None):
    m.eval(); out = []
    for i in range(0, len(idx), bs):
        with torch.autocast("cuda", enabled=(dev == "cuda")): o = m(gpu_batch(A, idx[i:i + bs], a=a))
        out.append(torch.sigmoid(o.float()).cpu())
    return torch.cat(out).numpy()
def load_arch(a, path):
    m = make_model(False, a); m.load_state_dict(torch.load(path, map_location=dev, weights_only=True)); return m.eval()
def val_scores(m, srcs):                                                             # SOURCE validation only, never the target
    return {k: mauc(SRC[k]["Y"][SRC[k]["va"]], predict(m, SRC[k]["arr"], SRC[k]["va"]), SRC[k]["valid"]) for k in srcs}

def batches(srcs, seed, ep):                                                         # natural pooled sampling
    rng = np.random.default_rng(seed * 100 + ep)
    dom = np.concatenate([np.full(len(SRC[k]["tr"]), i) for i, k in enumerate(srcs)]); pos = np.concatenate([SRC[k]["tr"] for k in srcs])
    perm = rng.permutation(len(pos))
    for k in range(math.ceil(len(pos) / BS)):
        s = perm[k * BS:(k + 1) * BS]
        yield {srcs[i]: np.sort(pos[s][dom[s] == i]) for i in range(len(srcs)) if (dom[s] == i).any()}
def batches_bal(srcs, seed, ep):                                                     # equal number of images from every hospital in every step
    rng = np.random.default_rng(seed * 100 + ep); h = BS // len(srcs)
    for _ in range(math.ceil(sum(len(SRC[k]["tr"]) for k in srcs) / BS)):
        yield {k: np.sort(rng.choice(SRC[k]["tr"], h, replace=False)) for k in srcs}
class GRL(torch.autograd.Function):                                                  # gradient reversal for DANN
    @staticmethod
    def forward(ctx, x, lam): ctx.lam = lam; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return -ctx.lam * g, None
def coral_loss(fa, fb):                                                              # match the feature covariances of the two hospitals
    d = fa.size(1); return ((torch.cov(fa.T) - torch.cov(fb.T)) ** 2).sum() / (4 * d * d)
def finish_and_save(m, srcs, BEST, PRD):                                             # predictions of the best checkpoint on every clean test site
    m.load_state_dict(torch.load(BEST, map_location=dev, weights_only=True)); z = {}
    z["A_test_P"] = predict(m, NIH_A, te_i); z["B_test_P"] = predict(m, CX_T, np.arange(len(CX_T)))
    if "C" not in srcs: z["C_test_P"] = predict(m, VB_A, np.arange(NVB))             # C is a clean test site only if it was not trained on
    for k in ["A", "B"] + (["C"] if "C" in srcs else []): z[f"{k}_val_P"] = predict(m, SRC[k]["arr"], SRC[k]["va"])
    np.savez(PRD, **z)
def _paths(a, ts, me, seed):
    b = stem(a, ts, me, seed); return b + ".pt", b + "_last.pt", b + "_history.csv", b + "_preds.npz"
def _resume(LAST, m, opt, sch, scaler, disc=None):
    ck = torch.load(LAST, map_location=dev, weights_only=False)
    m.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sch.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    if disc is not None and ck.get("disc") is not None: disc.load_state_dict(ck["disc"])
    print("resumed, next epoch", ck["ep"] + 2); return ck

In [ ]:
def train_vit_natural(ts, seed):                                                     # ViT-S ERM with natural pooled sampling
    a = "ViT-S"; use_arch(a); srcs = ts.split("+"); BEST, LAST, HIST, PRD = _paths(a, ts, NAT, seed)
    if os.path.exists(PRD): print("skip (finished):", os.path.basename(PRD)); return
    torch.manual_seed(seed); np.random.seed(seed)
    m = make_model(); opt = torch.optim.AdamW(m.parameters(), lr=CUR["lr"], weight_decay=CUR["wd"])
    spe = math.ceil(sum(len(SRC[k]["tr"]) for k in srcs) / BS); total, warm = EPOCHS * spe, min(300, spe)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: (t + 1) / warm if t < warm else 0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=(dev == "cuda")); hist, best, start = [], 0.0, 0
    if os.path.exists(LAST):
        ck = _resume(LAST, m, opt, sch, scaler); hist, best, start = ck["hist"], ck["best"], ck["ep"] + 1
    for ep in range(start, EPOCHS):
        m.train(); t0 = time.time(); tl = torch.zeros((), device=dev); n = 0
        for batch in tqdm(batches(srcs, seed, ep), total=spe, leave=False, desc=f"ViT {ts} ERM(nat) seed {seed} ep {ep + 1}/{EPOCHS}"):
            xs, ys, ms = [], [], []
            for k, idx in batch.items():
                xs.append(gpu_batch(SRC[k]["arr"], idx, flip=True)); ys.append(SRC[k]["Y"][idx]); ms.append(np.tile(SRC[k]["valid"], (len(idx), 1)))
            x = torch.cat(xs); y = torch.from_numpy(np.concatenate(ys)).to(dev).float(); mk = torch.from_numpy(np.concatenate(ms)).to(dev).float()
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=(dev == "cuda")): z = m(x)
            loss = (F.binary_cross_entropy_with_logits(z.float(), y, reduction="none") * mk).sum() / mk.sum()      # masked BCE: Edema is ignored for VinBigData
            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); tl += loss.detach() * len(x); n += len(x)
        vs = val_scores(m, srcs); vm = float(np.mean(list(vs.values())))
        row = dict(epoch=ep + 1, train_loss=float(tl) / n, val_mean=vm, **{f"val_{k}": v for k, v in vs.items()}, sec=time.time() - t0)
        hist.append(row); pd.DataFrame(hist).to_csv(HIST, index=False)
        print(f"ViT {ts} ERM(nat) seed {seed} ep {ep + 1} | loss {row['train_loss']:.4f} | val AUROC {vs} | {row['sec'] / 60:.1f} min")
        if vm > best: best = vm; torch.save(m.state_dict(), BEST)
        torch.save(dict(model=m.state_dict(), opt=opt.state_dict(), sched=sch.state_dict(), scaler=scaler.state_dict(), hist=hist, best=best, ep=ep), LAST)
    finish_and_save(m, srcs, BEST, PRD); del m, opt, sch, scaler; gc.collect(); torch.cuda.empty_cache()

def train_dg_vit(ts, meth, seed, lam_coral=1.0, lam_dann=0.1, eta_dro=0.01):         # ViT-S with ERM (balanced), DANN, CORAL or GroupDRO
    a = "ViT-S"; use_arch(a); assert meth != NAT and len(ts.split("+")) == 2
    srcs = ts.split("+"); BEST, LAST, HIST, PRD = _paths(a, ts, meth, seed)
    if os.path.exists(PRD): print("skip (finished):", os.path.basename(PRD)); return
    torch.manual_seed(seed); np.random.seed(seed)
    m = make_model(); disc = nn.Sequential(nn.Linear(m.num_features, 256), nn.ReLU(), nn.Linear(256, 2)).to(dev) if meth == "DANN" else None
    opt = torch.optim.AdamW(list(m.parameters()) + (list(disc.parameters()) if disc else []), lr=CUR["lr"], weight_decay=CUR["wd"])
    spe = math.ceil(sum(len(SRC[k]["tr"]) for k in srcs) / BS); total, warm = EPOCHS * spe, min(300, spe)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: (t + 1) / warm if t < warm else 0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=(dev == "cuda")); hist, best, start = [], 0.0, 0; q = torch.ones(2, device=dev) / 2
    if os.path.exists(LAST):
        ck = _resume(LAST, m, opt, sch, scaler, disc); hist, best, start, q = ck["hist"], ck["best"], ck["ep"] + 1, ck["q"].to(dev)
    for ep in range(start, EPOCHS):
        m.train(); t0 = time.time(); tl = torch.zeros((), device=dev); n = 0
        for step, batch in enumerate(tqdm(batches_bal(srcs, seed, ep), total=spe, leave=False, desc=f"ViT {ts} {meth} seed {seed} ep {ep + 1}/{EPOCHS}")):
            x = torch.cat([gpu_batch(SRC[k]["arr"], batch[k], flip=True) for k in srcs]); sizes = [len(batch[k]) for k in srcs]
            ys = [torch.from_numpy(SRC[k]["Y"][batch[k]]).to(dev).float() for k in srcs]
            mk = [torch.from_numpy(np.tile(SRC[k]["valid"], (len(batch[k]), 1))).to(dev).float() for k in srcs]
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=(dev == "cuda")):
                f = m.forward_head(m.forward_features(x), pre_logits=True); z = m.get_classifier()(f)
            z, f = z.float(), f.float(); zs, fs = torch.split(z, sizes), torch.split(f, sizes)
            gsum = [(F.binary_cross_entropy_with_logits(zz, yy, reduction="none") * mm).sum() for zz, yy, mm in zip(zs, ys, mk)]; gden = [mm.sum() for mm in mk]
            if meth == "GroupDRO":                                                   # exponentiated-gradient weights on the per-hospital losses
                gl = torch.stack([g_ / d_ for g_, d_ in zip(gsum, gden)]); q = q * torch.exp(eta_dro * gl.detach()); q = q / q.sum(); loss = (q * gl).sum()
            else:
                loss = sum(gsum) / sum(gden)                                         # masked pooled BCE (same as ERM)
                if meth == "CORAL": loss = loss + lam_coral * coral_loss(fs[0], fs[1])
                if meth == "DANN":                                                   # 0.1 x domain loss with the lambda ramp, as in your DenseNet runs
                    ramp = 2 / (1 + math.exp(-10 * (ep * spe + step) / total)) - 1
                    dom = torch.cat([torch.full((s_,), i, device=dev) for i, s_ in enumerate(sizes)])
                    loss = loss + lam_dann * F.cross_entropy(disc(GRL.apply(f, ramp)), dom)
            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); tl += loss.detach() * len(x); n += len(x)
        vs = val_scores(m, srcs); vm = float(np.mean(list(vs.values())))
        row = dict(epoch=ep + 1, train_loss=float(tl) / n, val_mean=vm, **{f"val_{k}": v for k, v in vs.items()}, sec=time.time() - t0)
        hist.append(row); pd.DataFrame(hist).to_csv(HIST, index=False)
        print(f"ViT {ts} {meth} seed {seed} ep {ep + 1} | loss {row['train_loss']:.4f} | val {vs}" + (f" | q={q.cpu().numpy().round(2)}" if meth == "GroupDRO" else ""))
        if vm > best: best = vm; torch.save(m.state_dict(), BEST)
        torch.save(dict(model=m.state_dict(), opt=opt.state_dict(), sched=sch.state_dict(), scaler=scaler.state_dict(), hist=hist, best=best, ep=ep, q=q.cpu(), **({"disc": disc.state_dict()} if disc else {})), LAST)
    finish_and_save(m, srcs, BEST, PRD); del m, opt, sch, scaler; gc.collect(); torch.cuda.empty_cache()

def train_dg_cnx(ts, meth, seed, dann_w=0.1, coral_w=1.0, dro_eta=0.01):             # ConvNeXt-T, recipe identical to your DenseNet runs
    a = "ConvNeXt-T"; use_arch(a); assert meth != NAT and len(ts.split("+")) == 2
    srcs = ts.split("+"); BEST, LAST, HIST, PRD = _paths(a, ts, meth, seed)
    if os.path.exists(PRD): print("skip (finished):", os.path.basename(PRD)); return
    torch.manual_seed(seed); np.random.seed(seed)
    m = make_model(); disc = nn.Sequential(nn.Linear(m.num_features, 256), nn.ReLU(), nn.Linear(256, len(srcs))).to(dev) if meth == "DANN" else None
    opt = torch.optim.AdamW(list(m.parameters()) + (list(disc.parameters()) if disc else []), lr=CUR["lr"], weight_decay=CUR["wd"])
    spe = math.ceil(sum(len(SRC[k]["tr"]) for k in srcs) / BS); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * spe)
    scaler = torch.amp.GradScaler("cuda", enabled=(dev == "cuda")); hist, best, start = [], 0.0, 0; q = torch.ones(len(srcs), device=dev) / len(srcs)
    if os.path.exists(LAST):
        ck = _resume(LAST, m, opt, sch, scaler, disc); hist, best, start, q = ck["hist"], ck["best"], ck["ep"] + 1, ck["q"].to(dev)
    tt = lambda arrs, dt: torch.from_numpy(np.concatenate(arrs)).to(dev).to(dt)
    for ep in range(start, EPOCHS):
        m.train(); t0 = time.time(); tl = torch.zeros((), device=dev)
        for k, batch in enumerate(tqdm(batches_bal(srcs, seed, ep), total=spe, leave=False, desc=f"ConvNeXt {ts} {meth} seed {seed} ep {ep + 1}/{EPOCHS}")):
            xs, ys, ms, ds = [], [], [], []
            for i, s_ in enumerate(srcs):
                idx = batch[s_]; xs.append(gpu_batch(SRC[s_]["arr"], idx, flip=True)); ys.append(SRC[s_]["Y"][idx])
                ms.append(np.tile(SRC[s_]["valid"], (len(idx), 1))); ds.append(np.full(len(idx), i))
            x = torch.cat(xs); y, mk, d = tt(ys, torch.float32), tt(ms, torch.float32), tt(ds, torch.long)
            with torch.autocast("cuda", enabled=(dev == "cuda")):
                f = m.forward_head(m.forward_features(x), pre_logits=True); z = m.get_classifier()(f)
            f = f.float(); ls = (F.binary_cross_entropy_with_logits(z.float(), y, reduction="none") * mk).sum(1) / mk.sum(1).clamp(min=1)   # per-sample loss, Edema masked for C
            if meth == "ERM (balanced)": loss = ls.mean()
            elif meth == "GroupDRO":
                gl = torch.stack([ls[d == i].mean() for i in range(len(srcs))]); q = q * torch.exp(dro_eta * gl.detach()); q = q / q.sum(); loss = (q * gl).sum()
            elif meth == "CORAL": loss = ls.mean() + coral_w * coral_loss(f[d == 0], f[d == 1])
            else:                                                                    # DANN: 0.1 x domain loss, lambda ramps 0 -> 1
                p = (ep * spe + k) / (EPOCHS * spe); lam = 2 / (1 + math.exp(-10 * p)) - 1
                loss = ls.mean() + dann_w * F.cross_entropy(disc(GRL.apply(f, lam)), d)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); tl += loss.detach()
        vs = val_scores(m, srcs); vm = float(np.mean(list(vs.values())))
        row = dict(epoch=ep + 1, train_loss=float(tl) / spe, val_mean=vm, **{f"val_{k}": v for k, v in vs.items()}, sec=time.time() - t0)
        hist.append(row); pd.DataFrame(hist).to_csv(HIST, index=False)
        print(f"ConvNeXt {ts} {meth} seed {seed} ep {ep + 1} | loss {row['train_loss']:.4f} | val {vs} | {row['sec'] / 60:.1f} min" + (f" | q={q.cpu().numpy().round(2)}" if meth == "GroupDRO" else ""))
        if vm > best: best = vm; torch.save(m.state_dict(), BEST)
        torch.save(dict(model=m.state_dict(), opt=opt.state_dict(), sched=sch.state_dict(), scaler=scaler.state_dict(), hist=hist, best=best, ep=ep, q=q.cpu(), **({"disc": disc.state_dict()} if disc else {})), LAST)
    finish_and_save(m, srcs, BEST, PRD); del m, opt, sch, scaler, disc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---------------- run plan: only what is missing is trained ----------------
PLAN = [("ViT-S", ts, NAT, s) for ts in TS for s in SEEDS] + \
       [("ViT-S", ts, me, s) for ts in SETS_ORDER for me in METHS[1:] for s in SEEDS] + \
       [("ConvNeXt-T", ts, me, s) for ts in SETS_ORDER for me in METHS[1:] for s in SEEDS]
isdone = lambda a, ts, me, s: os.path.exists(stem(a, ts, me, s) + "_preds.npz")
status = pd.DataFrame([dict(arch=a, train=ts, method=me, finished=sum(isdone(a, ts, me, s) for s in SEEDS), of=len(SEEDS)) for a in ["ViT-S", "ConvNeXt-T"] for ts in TS for me in METHS
                       if (a == "ViT-S" and (me == NAT or ts in POOLED)) or (a == "ConvNeXt-T" and me != NAT and ts in POOLED)])
print("seeds finished per model (ConvNeXt-T and DenseNet121 ERM natural are loaded from your earlier runs):"); display(status.pivot_table(index=["train", "method"], columns="arch", values="finished", fill_value=np.nan))
TODO = [p for p in PLAN if not isdone(*p)]; print(f"{len(TODO)} runs left of {len(PLAN)}")

RUN_TRAINING = True; STOP_AFTER = None            # STOP_AFTER = 3 trains three runs and stops; rerun the cell to continue
def run_one(a, ts, me, s):
    if a == "ViT-S": (train_vit_natural(ts, s) if me == NAT else train_dg_vit(ts, me, s))
    else: train_dg_cnx(ts, me, s)
if RUN_TRAINING:
    for a, ts, me, s in TODO[:STOP_AFTER]:
        try: run_one(a, ts, me, s)
        except torch.cuda.OutOfMemoryError:
            print("GPU out of memory: set BS = 32 in the setup cell, restart the kernel, rerun the setup cells, then rerun this cell (it resumes)."); break
print("all finished" if not [p for p in PLAN if not isdone(*p)] else "runs left: rerun this cell")

# Part C: load every prediction

All three architectures are put on the same footing: for every (architecture, training set, method, seed) a dictionary `{test site: probabilities}`. A test site is only listed when it was **not** a training source for that model, plus A and B for models that did train on them (marked as *seen*).

In [ ]:
FN = {"DenseNet121": "DenseNet", "ConvNeXt-T": "ConvNeXt"}
TAGF = {NAT: "erm_nat", "ERM (balanced)": "erm_bal", "DANN": "dann", "CORAL": "coral"}         # tags used by your full11 logits (no GroupDRO there)
def load_probs(a, ts, me, s):
    if a == "ViT-S" or (a == "ConvNeXt-T" and me != NAT):                                       # models trained by this notebook: probabilities
        f = stem(a, ts, me, s) + "_preds.npz"
        if not os.path.exists(f): return None
        z = np.load(f); return {T: z[f"{T}_test_P"] for T in "ABC" if f"{T}_test_P" in z}
    tag = TAGF.get(me)                                                                          # DenseNet121 and ConvNeXt-T ERM (natural): logits from fair_ens
    if tag is None: return None                                                                 # DenseNet GroupDRO has no VinBigData predictions
    f = os.path.join(FE, f"logits_{FN[a]}_{ts.replace('+', 'p')}_{tag}_seed{s}.npz")
    if not os.path.exists(f): return None
    z = np.load(f); return {T: sig(z[T]) for T in "ABC" if not (T == "C" and "C" in ts.split("+"))}

PR3 = {}
for a in A3:
    for ts in TS:
        for me in METHS:
            d = {s: load_probs(a, ts, me, s) for s in SEEDS}; d = {s: v for s, v in d.items() if v is not None}
            if d: PR3[(a, ts, me)] = d
cov = pd.DataFrame([dict(arch=a, train=ts, method=me, seeds=len(d)) for (a, ts, me), d in PR3.items()]).pivot_table(index=["train", "method"], columns="arch", values="seeds", fill_value=0)
print("seeds available per model (0 = not available):"); display(cov); save_tab(cov, "table_coverage")

EXT   = [(ts, T) for ts in TS for T in "ABC" if T not in ts.split("+")]       # 9 external directions
SEENC = [(ts, T) for ts in TS for T in "AB" if T in ts.split("+")]            # in-distribution cells that have a clean held-out test set
ALLC  = EXT + SEENC
def pm(a, ts, T, me=NAT): d = PR3[(a, ts, me)]; return np.mean([d[s][T] for s in d], 0)       # seed-averaged probabilities
for a in A3: assert all((a, ts, NAT) in PR3 for ts in TS), f"ERM (natural) predictions of {a} are incomplete: check the paths in Part A"

# Part D: main results

**Table 4** gives the AUROC and AUPRC of every training set on every test site (mean ± sd over seeds). A dagger marks a test site that was part of the training set. Column C uses five findings (no Edema). **Fig. 2** shows the three transfer matrices and the pairwise architecture differences.

In [ ]:
rows = []
for a in A3:
    for ts, T in ALLC:
        if (a, ts, NAT) not in PR3: continue
        d = PR3[(a, ts, NAT)]
        v = [mauc(YT[T], d[s][T], VALID[T]) for s in d]; p = [mauprc(YT[T], d[s][T], VALID[T]) for s in d]
        rows.append(dict(arch=a, train=ts, test=T, seen=T in ts.split("+"),
                         auroc=np.mean(v), auroc_sd=np.std(v, ddof=1) if len(v) > 1 else 0,
                         auprc=np.mean(p), auprc_sd=np.std(p, ddof=1) if len(p) > 1 else 0))
R3 = pd.DataFrame(rows); save_tab(R3, "table4_main_numeric", index=False)
for m in ["auroc", "auprc"]:
    t = R3.assign(v=R3.apply(lambda r: f"{r[m]:.3f} ± {r[m + '_sd']:.3f}" + (" †" if r.seen else ""), axis=1)).pivot(index="train", columns=["test", "arch"], values="v").reindex(TS)
    print(f"\n=== {m.upper()} by architecture (ERM natural, mean ± sd over seeds; † = seen site; column C = 5 findings) ==="); display(t); save_tab(t, f"table4_main_{m}")

M = {a: np.array([[R3[(R3.arch == a) & (R3.train == ts) & (R3.test == T)].auroc.mean() if len(R3[(R3.arch == a) & (R3.train == ts) & (R3.test == T)]) else np.nan for T in "ABC"] for ts in TS]) for a in A3}
fig, axes = plt.subplots(2, 3, figsize=(COL2, 5.6)); lo_, hi_ = np.nanmin(list(M.values())) - .02, np.nanmax(list(M.values())) + .01
for a_, a in zip(axes[0], A3):
    a_.imshow(M[a], cmap="Greys", vmin=lo_, vmax=hi_)
    for i, ts in enumerate(TS):
        for j, T in enumerate("ABC"):
            if not np.isnan(M[a][i, j]): a_.text(j, i, f"{M[a][i, j]:.3f}" + ("†" if T in ts.split("+") else ""), ha="center", va="center", fontsize=6, color="w" if M[a][i, j] > (lo_ + hi_) / 2 else "k")
    a_.set_title(a); a_.set_xticks(range(3)); a_.set_xticklabels(["A", "B", "C*"]); a_.set_yticks(range(len(TS))); a_.set_yticklabels(TS if a_ is axes[0, 0] else [""] * len(TS)); a_.grid(False)
for a_, (b, a) in zip(axes[1], [("ConvNeXt-T", "DenseNet121"), ("ViT-S", "DenseNet121"), ("ViT-S", "ConvNeXt-T")]):
    D = M[b] - M[a]; mx = np.nanmax(np.abs(D)) + 1e-6; im2 = a_.imshow(D, cmap="RdBu", vmin=-mx, vmax=mx)
    for i in range(len(TS)):
        for j in range(3):
            if not np.isnan(D[i, j]): a_.text(j, i, f"{D[i, j]:+.3f}", ha="center", va="center", fontsize=6)
    a_.set_title(f"{b} − {a}"); a_.set_xticks(range(3)); a_.set_xticklabels(["A", "B", "C*"]); a_.set_yticks(range(len(TS))); a_.set_yticklabels(TS if a_ is axes[1, 0] else [""] * len(TS)); a_.grid(False)
    fig.colorbar(im2, ax=a_, shrink=.7)
fig.suptitle("Top: AUROC (rows = training set, columns = test site; † seen, *5 findings). Bottom: architecture differences", fontsize=7); fig.tight_layout(); save_fig(fig, "fig02_transfer_matrices")

# Part E: statistics

**Protocol.** One patient-level bootstrap resample per test site is drawn in every replicate and shared by *all* models, so every difference is a **paired** difference. VinBigData has no patient identifier, so its resampling is at image level. P-values are two-sided bootstrap p-values; their smallest attainable value is `1/NB`. With `NB = 2000` the Holm correction over about 45 comparisons can still reach significance. With `NB = 300` it cannot, and a "no significant difference" statement would be a resolution artifact, not a finding.

In [ ]:
rng = np.random.default_rng(11)
cells3 = [(a, ts, T) for a in A3 for ts, T in ALLC if (a, ts, NAT) in PR3]
PM3 = {c: pm(*c) for c in cells3}; BL = {c: np.zeros((NB, L)) for c in cells3}
for r in tqdm(range(NB), desc="paired patient bootstrap"):
    ix = {T: resample(T, rng) for T in "ABC"}                      # one resample per site, shared by all architectures = paired
    for (a, ts, T) in cells3: BL[(a, ts, T)][r] = lab_auc(YT[T][ix[T]], PM3[(a, ts, T)][ix[T]], VALID[T])
BM = {c: np.nanmean(v, 1) for c, v in BL.items()}
PTL3 = {c: lab_auc(YT[c[2]], PM3[c], VALID[c[2]]) for c in cells3}; PTM3 = {c: np.nanmean(v) for c, v in PTL3.items()}

rows = [dict(arch=a, train=ts, test=T, seen=T in ts.split("+"), auroc=PTM3[(a, ts, T)], lo=pci(BM[(a, ts, T)])[0], hi=pci(BM[(a, ts, T)])[1]) for (a, ts, T) in cells3]
CI3 = pd.DataFrame(rows); save_tab(CI3, "table5_auroc_with_ci", index=False); display(CI3.round(3).head(12))

GP3, GB3 = {}, {}
for a in A3:                                                      # gap = in-hospital AUROC minus external AUROC, six findings, A and B only
    if not all(c in BL for c in [(a, "A", "A"), (a, "A", "B"), (a, "B", "B"), (a, "B", "A")]): print("skipped (missing models):", a); continue
    GP3[a] = dict(ab=PTL3[(a, "A", "A")] - PTL3[(a, "A", "B")], ba=PTL3[(a, "B", "B")] - PTL3[(a, "B", "A")])
    GB3[a] = dict(ab=BL[(a, "A", "A")] - BL[(a, "A", "B")], ba=BL[(a, "B", "B")] - BL[(a, "B", "A")])
    GP3[a]["asym"] = GP3[a]["ab"] - GP3[a]["ba"]; GB3[a]["asym"] = GB3[a]["ab"] - GB3[a]["ba"]
QN = {"ab": "Gap A→B (NIH→CheXpert)", "ba": "Gap B→A (CheXpert→NIH)", "asym": "Asymmetry (A→B gap − B→A gap)"}
rows = []
for a in GP3:
    for k, nm in QN.items():
        b = np.nanmean(GB3[a][k], 1); rows.append(dict(arch=a, quantity=nm, estimate=np.nanmean(GP3[a][k]), lo=np.percentile(b, 2.5), hi=np.percentile(b, 97.5)))
GT = pd.DataFrame(rows); display(GT.round(3)); save_tab(GT, "table6_gaps_asymmetry", index=False)
arch_pairs = [("ConvNeXt-T", "DenseNet121"), ("ViT-S", "DenseNet121"), ("ViT-S", "ConvNeXt-T")]
rows = []
for b_, a_ in arch_pairs:
    if b_ in GP3 and a_ in GP3:
        d = np.nanmean(GB3[b_]["asym"], 1) - np.nanmean(GB3[a_]["asym"], 1)
        rows.append(dict(pair=f"{b_} − {a_}", diff_in_asymmetry=np.nanmean(GP3[b_]["asym"]) - np.nanmean(GP3[a_]["asym"]), lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5)))
DA = pd.DataFrame(rows); DA["CI_includes_0"] = (DA.lo <= 0) & (DA.hi >= 0); display(DA.round(3)); save_tab(DA, "table7_asymmetry_differences", index=False)

fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.9), gridspec_kw=dict(width_ratios=[1, 1.5])); w = .26
for i, a in enumerate(GP3):
    g = GT[GT.arch == a]; ax[0].bar(np.arange(3) + (i - 1) * w, g.estimate, w, yerr=[np.clip(g.estimate - g.lo, 0, None), np.clip(g.hi - g.estimate, 0, None)], capsize=2, color=AC[a], edgecolor="k", lw=.4, label=a)
ax[0].set_xticks(range(3)); ax[0].set_xticklabels(["Gap A→B", "Gap B→A", "Asymmetry"]); ax[0].axhline(0, color="k", lw=.6); ax[0].set(ylabel="Mean AUROC difference"); ax[0].legend(frameon=False); ax[0].grid(True, axis="y", alpha=.3)
x = np.arange(L)
for i, a in enumerate(GP3):
    for o, k, hatch in [(-.04, "ab", ""), (.04, "ba", "///")]:
        pt_ = GP3[a][k]; c = pci(GB3[a][k]); ax[1].bar(x + (i - 1) * .27 + o * 2.2, pt_, .12, yerr=[np.clip(pt_ - c[0], 0, None), np.clip(c[1] - pt_, 0, None)], capsize=1, color=AC[a], hatch=hatch, edgecolor="k", lw=.3)
ax[1].set_xticks(x); ax[1].set_xticklabels(LABELS6, rotation=35, ha="right"); ax[1].axhline(0, color="k", lw=.6); ax[1].set(ylabel="Per-label AUROC drop"); ax[1].set_title("Per label (solid: A→B, hatched: B→A)", fontsize=7); ax[1].grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig03_gaps_asymmetry")

In [ ]:
# ---------------- architecture differences, Holm-corrected over all comparisons ----------------
rows = []
for b_, a_ in arch_pairs:
    for ts, T in ALLC:
        if (b_, ts, T) in BM and (a_, ts, T) in BM:
            d = BM[(b_, ts, T)] - BM[(a_, ts, T)]
            rows.append(dict(pair=f"{b_} − {a_}", direction=dname(ts, T), seen=T in ts.split("+"), diff=PTM3[(b_, ts, T)] - PTM3[(a_, ts, T)], lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5), p=boot_p(d, NB)))
PD3 = pd.DataFrame(rows); PD3["p_holm"] = holm(PD3.p.values); PD3["significant"] = PD3.p_holm < .05
print(f"Holm over {len(PD3)} comparisons; smallest attainable raw p = 1/NB = {1 / NB:.4f}; smallest attainable Holm p = {len(PD3) / NB:.4f}")
if len(PD3) / NB >= .05: print("WARNING: NB is too small for this many comparisons: significance cannot be reached. Set FAST = False.")
display(PD3.round(4)); save_tab(PD3, "table8_architecture_pairwise", index=False)

dirs = [dname(ts, T) for ts, T in ALLC]; yy = {d: i for i, d in enumerate(dirs)}; mk = {"ConvNeXt-T − DenseNet121": "o", "ViT-S − DenseNet121": "s", "ViT-S − ConvNeXt-T": "^"}
fig, ax = plt.subplots(figsize=(COL2, 4.6))
for k, (pair, g) in enumerate(PD3.groupby("pair")):
    for _, r in g.iterrows():
        ax.errorbar(r["diff"], yy[r.direction] + (k - 1) * .22, xerr=[[r["diff"] - r.lo], [r.hi - r["diff"]]], fmt=mk[pair], color="k", mfc="k" if r.significant else "w", ms=3.5, capsize=1.5, lw=.7, label=pair if r.direction == dirs[0] else None)
ax.axvline(0, color="k", ls="--", lw=.6); ax.axhline(len(EXT) - .5, color="0.6", lw=.6); ax.set_yticks(range(len(dirs))); ax.set_yticklabels([d + ("  (seen)" if i >= len(EXT) else "") for i, d in enumerate(dirs)]); ax.invert_yaxis()
ax.set(xlabel="AUROC difference (95% CI; filled = significant after Holm)"); ax.legend(frameon=False, fontsize=6, loc="lower right"); ax.grid(True, axis="x", alpha=.3); fig.tight_layout(); save_fig(fig, "fig04_architecture_forest")

fig, ax = plt.subplots(figsize=(COL2, 3.0)); w = .26
for i, a in enumerate(A3):
    v = np.array([PTM3.get((a, ts, T), np.nan) for ts, T in EXT]); c = np.array([pci(BM[(a, ts, T)]) if (a, ts, T) in BM else [np.nan, np.nan] for ts, T in EXT])
    ax.bar(np.arange(len(EXT)) + (i - 1) * w, v, w, yerr=[np.clip(v - c[:, 0], 0, None), np.clip(c[:, 1] - v, 0, None)], capsize=1, color=AC[a], edgecolor="k", lw=.3, label=a)
ax.set_xticks(range(len(EXT))); ax.set_xticklabels([dname(*e) for e in EXT], rotation=40, ha="right"); ax.set_ylim(.55, None); ax.set(ylabel="External macro AUROC"); ax.legend(frameon=False); ax.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig05_external_auroc")

fig, axes = plt.subplots(1, 3, figsize=(COL2, 3.4), sharey=True)                                  # per-disease external AUROC per architecture
for a_, a in zip(axes, A3):
    D = np.array([PTL3[(a, ts, T)] for ts, T in EXT]); im = a_.imshow(D, cmap="viridis", vmin=.5, vmax=.95, aspect="auto")
    for i in range(D.shape[0]):
        for j in range(L): a_.text(j, i, "–" if np.isnan(D[i, j]) else f"{D[i, j]:.2f}", ha="center", va="center", color="w", fontsize=5)
    a_.set_xticks(range(L)); a_.set_xticklabels([l.replace("Pleural ", "Pl. ") for l in LABELS6], rotation=45, ha="right", fontsize=6); a_.set_title(a); a_.grid(False)
    a_.set_yticks(range(len(EXT))); a_.set_yticklabels([dname(*e) for e in EXT] if a_ is axes[0] else [""] * len(EXT))
fig.colorbar(im, ax=axes, label="AUROC", shrink=.8); save_fig(fig, "fig06_per_disease_external")

# Part F: what explains the cross-hospital gap?

Three candidate explanations are examined with the saved predictions:

1. **Case mix (view position).** CheXpert test images are mostly AP and NIH test images are about 57% AP, and disease prevalence differs strongly by view (for example Edema is about 30% in CheXpert AP and 6% in CheXpert PA). If the gap shrinks when only PA or only AP images are scored, case mix explains part of it.
2. **Prevalence (label) shift.** Compared in Part G with a label-free correction.
3. **Training-set size.** Your size-matched control (CheXpert-77k vs NIH-77k, DenseNet121) is in `che44-nih77.ipynb`. It is not repeated here because those checkpoints are DenseNet-only. In the saved copy of that notebook the last bootstrap cell stopped at 46%, so rerun it once and cite its table.

These are attributions by conditioning, not a causal decomposition, and the terms can interact. Say so in the paper.

In [ ]:
if VIEW is None:
    print("view columns missing: this analysis is skipped")
else:
    VIEWS = ["PA", "AP"]
    OKV = {v: [j for j in range(L) if min(min(YT[T][VIEW[T] == v, j].sum(), (VIEW[T] == v).sum() - YT[T][VIEW[T] == v, j].sum()) for T in "AB") >= 30] for v in VIEWS}   # findings with >= 30 positives and negatives in both sites
    vvalid = {v: np.isin(np.arange(L), OKV[v]) for v in VIEWS}
    print({v: [LABELS6[j] for j in OKV[v]] for v in VIEWS})
    prv = pd.concat({f"{SITE[T]} ({T})": pd.DataFrame(YT[T], columns=LABELS6).groupby(VIEW[T]).mean() for T in "AB"}).round(3); display(prv); save_tab(prv, "table9_prevalence_by_view")

    def vauc(a, ts, T, v, ix):
        sel = ix[VIEW[T][ix] == v]; return np.nanmean(lab_auc(YT[T][sel], PM3[(a, ts, T)][sel], vvalid[v]))
    full = {T: np.arange(len(YT[T])) for T in "AB"}
    def vgaps(a, ixA, ixB, v):
        ab = vauc(a, "A", "A", v, ixA) - vauc(a, "A", "B", v, ixB); ba = vauc(a, "B", "B", v, ixB) - vauc(a, "B", "A", v, ixA); return ab, ba
    BV = {(a, v, k): np.zeros(NB_VIEW) for a in A3 for v in VIEWS for k in ("ab", "ba", "asym")}; rngv = np.random.default_rng(31)
    for r in tqdm(range(NB_VIEW), desc="view-stratified bootstrap"):
        ix = {T: resample(T, rngv) for T in "AB"}
        for a in A3:
            for v in VIEWS:
                ab, ba = vgaps(a, ix["A"], ix["B"], v); BV[(a, v, "ab")][r], BV[(a, v, "ba")][r], BV[(a, v, "asym")][r] = ab, ba, ab - ba
    rows = []
    for a in A3:
        for v in VIEWS:
            ab, ba = vgaps(a, full["A"], full["B"], v)
            for k, pt in [("ab", ab), ("ba", ba), ("asym", ab - ba)]:
                rows.append(dict(arch=a, view=v, quantity={"ab": "Gap A→B", "ba": "Gap B→A", "asym": "Asymmetry"}[k], estimate=pt, lo=np.percentile(BV[(a, v, k)], 2.5), hi=np.percentile(BV[(a, v, k)], 97.5)))
        for k, nm in [("ab", "Gap A→B"), ("ba", "Gap B→A"), ("asym", "Asymmetry")]:
            b = np.nanmean(GB3[a][k], 1); rows.append(dict(arch=a, view="All", quantity=nm, estimate=np.nanmean(GP3[a][k]), lo=np.percentile(b, 2.5), hi=np.percentile(b, 97.5)))
    VG = pd.DataFrame(rows); display(VG.round(3)); save_tab(VG, "table10_view_matched_gaps", index=False)
    print("If the asymmetry stays clearly above 0 in BOTH views, case mix (PA/AP) cannot be the whole explanation.")

    fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.8), sharey=True); order = ["All", "PA", "AP"]; w = .26
    for a_, q in zip(ax, ["Gap A→B", "Gap B→A", "Asymmetry"]):
        for i, a in enumerate(A3):
            g = VG[(VG.arch == a) & (VG.quantity == q)].set_index("view").reindex(order)
            a_.bar(np.arange(3) + (i - 1) * w, g.estimate, w, yerr=[np.clip(g.estimate - g.lo, 0, None), np.clip(g.hi - g.estimate, 0, None)], capsize=1.5, color=AC[a], edgecolor="k", lw=.4, label=a)
        a_.set_xticks(range(3)); a_.set_xticklabels(order); a_.axhline(0, color="k", lw=.6); a_.set_title(q); a_.grid(True, axis="y", alpha=.3)
    ax[0].set_ylabel("Mean AUROC difference"); ax[0].legend(frameon=False, fontsize=6); fig.tight_layout(); save_fig(fig, "fig07_view_matched_gaps")

# Part G: calibration, label-free prior-shift correction, k-label recalibration

* **G1** ECE, Brier score and calibration-in-the-large (mean predicted probability minus observed prevalence) for every external direction.
* **G2** The **k-label** correction: Platt scaling and a re-tuned F1 threshold fitted on k labelled target-validation images (k = 0 ... 5000, 20 random draws), for A→B and B→A in all three architectures. These corrections are monotone, so **AUROC cannot change**: they can repair calibration and thresholds, not ranking quality.
* **G3** A **label-free** correction (EM prior-shift adjustment, Saerens et al.) that needs no target labels. It assumes calibrated source posteriors, which deep networks often lack, so it is run three ways: plain EM, source-calibration only, and source-calibration followed by EM (Alexandari et al.). Treat it as a baseline, not a guarantee.

In [ ]:
rows = []
for a in A3:
    for ts, T in ALLC:
        if (a, ts, NAT) not in PR3: continue
        v = np.where(VALID[T])[0]; Y = YT[T]
        for s, d in PR3[(a, ts, NAT)].items():
            P = d[T]; rows.append(dict(arch=a, train=ts, test=T, seen=T in ts.split("+"), seed=s, ece=np.mean([ece_of(Y[:, j], P[:, j]) for j in v]), brier=np.mean((P[:, v] - Y[:, v]) ** 2),
                                       citl=np.mean([P[:, j].mean() - Y[:, j].mean() for j in v])))
CA = pd.DataFrame(rows); save_tab(CA, "table11_calibration_numeric", index=False)
for m in ["ece", "brier", "citl"]:
    t = CA.groupby(["train", "test", "arch"])[m].agg(lambda x: f"{x.mean():+.3f} ± {x.std(ddof=1) if len(x) > 1 else 0:.3f}" if m == "citl" else f"{x.mean():.3f} ± {x.std(ddof=1) if len(x) > 1 else 0:.3f}").unstack("arch")
    print(f"\n=== {m.upper()} ({'positive = over-prediction' if m == 'citl' else 'lower is better'}) ==="); display(t); save_tab(t, f"table11_calibration_{m}")
fig, ax = plt.subplots(1, 2, figsize=(COL2, 3.0)); w = .26
for a_, m, t in zip(ax, ["ece", "brier"], ["ECE (external directions)", "Brier score (external directions)"]):
    for i, a in enumerate(A3):
        g = [CA[(CA.arch == a) & (CA.train == ts) & (CA.test == T)][m] for ts, T in EXT]
        a_.bar(np.arange(len(EXT)) + (i - 1) * w, [x.mean() if len(x) else np.nan for x in g], w, yerr=[x.std(ddof=1) if len(x) > 1 else 0 for x in g], capsize=1, color=AC[a], edgecolor="k", lw=.3, label=a)
    a_.set_xticks(range(len(EXT))); a_.set_xticklabels([dname(*e) for e in EXT], rotation=40, ha="right"); a_.set_title(t); a_.grid(True, axis="y", alpha=.3)
ax[0].legend(frameon=False); fig.tight_layout(); save_fig(fig, "fig08_calibration_bars")
fig, axes = plt.subplots(1, 3, figsize=(COL2, 2.7), sharey=True)
for a_, (ts, T) in zip(axes, [("A", "B"), ("B", "A"), ("A+B", "C")]):
    v = np.where(VALID[T])[0]
    for a in A3:
        if (a, ts, NAT) in PR3: xs_, ys_ = reliab(YT[T][:, v].ravel(), pm(a, ts, T)[:, v].ravel()); a_.plot(xs_, ys_, ls=ALS[a], marker="o", ms=2.5, color=AC[a] if a != "ViT-S" else "0.5", label=a)
    a_.plot([0, 1], [0, 1], "k:", lw=.6); a_.set(title=dname(ts, T), xlabel="Predicted probability"); a_.grid(True, alpha=.3)
axes[0].set_ylabel("Observed frequency"); axes[0].legend(frameon=False, fontsize=6); fig.tight_layout(); save_fig(fig, "fig09_reliability")

In [ ]:
# ---------------- G2: k-label recalibration (A->B and B->A) for all three architectures ----------------
XDIR = os.path.join(MASTER, "extra"); os.makedirs(XDIR, exist_ok=True)
def single_ckpt(a, src, s):                                                        # checkpoints of the single-source models trained earlier
    if a == "ViT-S": return stem("ViT-S", src, NAT, s) + ".pt"
    if a == "DenseNet121": return os.path.join(OUT, "nih_train", f"densenet121_nih_e{EPOCHS}_seed{s}.pt") if src == "A" else os.path.join(OUT, f"densenet121_e{EPOCHS}_seed{s}.pt")
    return os.path.join(OUT, "part34", f"convnext_tiny_{'nih' if src == 'A' else 'cx'}_e{EPOCHS}_seed{s}.pt")
def extra_preds(a, src, s):                                                        # source-validation and target-validation (pool) probabilities, cached
    f = os.path.join(XDIR, f"extra_{a}_{src}_seed{s}.npz"); tgt = "B" if src == "A" else "A"
    if os.path.exists(f): return dict(np.load(f))
    if a == "ViT-S":
        z = np.load(stem("ViT-S", src, NAT, s) + "_preds.npz"); out = {"src_val": z[f"{src}_val_P"], "tgt_val": z[f"{tgt}_val_P"]}
    else:
        ck = single_ckpt(a, src, s)
        if not os.path.exists(ck): print("missing checkpoint, skipped:", ck); return None
        try: m = load_arch(a, ck)
        except Exception as e: print(f"could not load {os.path.basename(ck)} as a timm {a} (different architecture code?): {str(e)[:120]}"); return None
        out = {"src_val": predict(m, SRC[src]["arr"], SRC[src]["va"], a=a), "tgt_val": predict(m, SRC[tgt]["arr"], SRC[tgt]["va"], a=a)}; del m
    np.savez(f, **out); return out
def fit_platt(z, y):
    lr = LogisticRegression(C=1e6, max_iter=500).fit(z.reshape(-1, 1), y); return float(np.clip(lr.coef_[0, 0], .1, 5)), float(lr.intercept_[0])
def correct(meth, zp, yp, ze, t0):
    a_, b_, t = 1.0, 0.0, t0; npos = int(yp.sum()); nneg = len(yp) - npos
    if npos >= 5 and nneg >= 5:                                                    # too few positives: fall back to the uncorrected model
        if meth == "Platt + threshold": a_, b_ = fit_platt(zp, yp)
        t = best_thr(yp, sig(a_ * zp + b_))
    return sig(a_ * ze + b_), t
KS, DRAWS, RMETH = [0, 25, 50, 100, 250, 500, 1000, 2500, 5000], 20, ["Threshold only", "Platt + threshold"]
rows, ref = [], []
for a in A3:
    for src in ["A", "B"]:
        T = "B" if src == "A" else "A"
        if (a, src, NAT) not in PR3: continue
        for s in PR3[(a, src, NAT)]:
            ex = extra_preds(a, src, s)
            if ex is None: continue
            Pe, Ye = PR3[(a, src, NAT)][s][T], YT[T]; Ze = logit(Pe); Zp = logit(ex["tgt_val"]); Yp = SRC[T]["Y"][SRC[T]["va"]]
            Ysv = SRC[src]["Y"][SRC[src]["va"]]; thr0 = np.array([best_thr(Ysv[:, j], ex["src_val"][:, j]) for j in range(L)])
            Pi, Yi = PR3[(a, src, NAT)][s][src], YT[src]
            ref.append(dict(arch=a, src=src, seed=s, ece_in=np.mean([ece_of(Yi[:, j], Pi[:, j]) for j in range(L)]), f1_in=np.mean([f1_at(Yi[:, j], Pi[:, j], thr0[j]) for j in range(L)]),
                            ece_0=np.mean([ece_of(Ye[:, j], Pe[:, j]) for j in range(L)]), f1_0=np.mean([f1_at(Ye[:, j], Pe[:, j], thr0[j]) for j in range(L)]), auroc_ext=mauc(Ye, Pe, VALID[T])))
            for K in KS:
                for dr in range(1 if K == 0 else DRAWS):
                    idx = np.random.default_rng([s, K, dr]).choice(len(Yp), min(K, len(Yp)), replace=False) if K else np.array([], int)
                    for me in RMETH:
                        es, fs = [], []
                        for j in range(L):
                            pe, t = correct(me, Zp[idx, j], Yp[idx, j], Ze[:, j], thr0[j]) if K else (sig(Ze[:, j]), thr0[j])
                            es.append(ece_of(Ye[:, j], pe)); fs.append(f1_at(Ye[:, j], pe, t))
                        rows.append(dict(arch=a, src=src, seed=s, method=me, K=K, draw=dr, ece=np.mean(es), f1=np.mean(fs)))
RC, RF = pd.DataFrame(rows), pd.DataFrame(ref)
if len(RC) == 0: print("no recalibration inputs found: check the checkpoint paths in single_ckpt")
else:
    save_tab(RC, "table12_recalibration_raw", index=False); psr = RC.groupby(["arch", "src", "method", "K", "seed"])[["ece", "f1"]].mean().reset_index(); refm = RF.groupby(["arch", "src"]).mean(numeric_only=True)
    tab = psr[psr.K.isin([0, 100, 500, 2500])].groupby(["arch", "src", "method", "K"])[["ece", "f1"]].mean().round(3).unstack("K"); display(tab); save_tab(tab, "table12_recalibration_summary")
    rec = psr[(psr.method == "Platt + threshold") & psr.K.isin([100, 500, 2500])].merge(RF, on=["arch", "src", "seed"])
    rec["F1 gap recovered"] = np.where((rec.f1_in - rec.f1_0).abs() > .01, (rec.f1 - rec.f1_0) / (rec.f1_in - rec.f1_0), np.nan)
    rec["ECE gap recovered"] = np.where((rec.ece_0 - rec.ece_in).abs() > .01, (rec.ece_0 - rec.ece) / (rec.ece_0 - rec.ece_in), np.nan)
    rect = (rec.groupby(["arch", "src", "K"])[["F1 gap recovered", "ECE gap recovered"]].mean() * 100).round(0); display(rect); save_tab(rect, "table13_recalibration_recovery_percent")
    print("AUROC before and after is identical (monotone correction):"); display(RF.groupby(["arch", "src"])[["auroc_ext"]].mean().round(3))
    DNAME = {"A": "A→B (NIH→CheXpert)", "B": "B→A (CheXpert→NIH)"}
    fig, ax = plt.subplots(2, 2, figsize=(COL2, 5.0))
    for r_, src in enumerate(["A", "B"]):
        for c_, (m, ttl) in enumerate([("ece", "ECE (lower is better)"), ("f1", "Macro-F1 (higher is better)")]):
            for a in A3:
                for me, lw_, al in [("Platt + threshold", 1.2, 1.0), ("Threshold only", .7, .55)]:
                    g = psr[(psr.arch == a) & (psr.src == src) & (psr.method == me)].groupby("K")[m].mean()
                    if len(g): ax[r_, c_].plot(g.index, g.values, ls=ALS[a], marker=MKR[a], ms=2.5, lw=lw_, alpha=al, color="k" if me.startswith("Platt") else "0.5", label=f"{a}, {me}" if (r_, c_) == (0, 0) else None)
                if (a, src) in refm.index: ax[r_, c_].axhline(refm.loc[(a, src), m + "_in"], color=AC[a], ls=ALS[a], lw=.5)
            ax[r_, c_].set_xscale("symlog", linthresh=25); ax[r_, c_].set_xlim(left=0); ax[r_, c_].set(xlabel="k labelled target images", title=f"{DNAME[src]}: {ttl}"); ax[r_, c_].grid(True, alpha=.3)
    h, l_ = ax[0, 0].get_legend_handles_labels(); fig.legend(h, l_, loc="lower center", ncol=3, frameon=False, fontsize=5.5); fig.tight_layout(rect=[0, .07, 1, 1]); save_fig(fig, "fig10_recalibration_vs_k")

# Part H: domain generalization methods vs ERM

For each architecture, ERM (natural), ERM (balanced), DANN, CORAL and GroupDRO are compared on the **unseen** hospital of each pooled set (A+B → C, A+C → B, B+C → A).

**Two baselines are used, because the sampling scheme is itself a variable.** DANN, CORAL and GroupDRO all train with balanced sampling (equal images per hospital per step), while ERM (natural) samples proportionally to hospital size. A method beating ERM (natural) could therefore be winning because of balanced sampling alone. The **fair baseline for a DG method is ERM (balanced)**. Both are reported, each with its own Holm family.

In [ ]:
# ---------------- G3: label-free prior-shift correction (needs no target labels) ----------------
# Plain Saerens EM assumes calibrated source posteriors. Deep networks are not, and then EM can run away. Following Alexandari et al. (2020), the
# posteriors are first calibrated on the SOURCE validation set (Platt), then EM estimates the target prevalence. Three variants are compared.
def saerens(p, pi_s, iters=300, tol=1e-7):
    p = np.clip(p, 1e-6, 1 - 1e-6); pi_t = pi_s
    for _ in range(iters):
        num = p * (pi_t / pi_s); q = num / (num + (1 - p) * ((1 - pi_t) / (1 - pi_s))); new = float(np.clip(q.mean(), 1e-4, 1 - 1e-4))
        if abs(new - pi_t) < tol: break
        pi_t = new
    return q, pi_t
rows = []
for a in A3:
    for src in ["A", "B"]:
        T = "B" if src == "A" else "A"
        if (a, src, NAT) not in PR3: continue
        Y = YT[T]; Ysv = SRC[src]["Y"][SRC[src]["va"]]; prev_t = Y.mean(0)
        for s in PR3[(a, src, NAT)]:
            ex = extra_preds(a, src, s)
            if ex is None: continue
            P = PR3[(a, src, NAT)][s][T]; Q_raw, Q_cal, Q_em = P.copy(), P.copy(), P.copy(); est_raw, est_em = np.zeros(L), np.zeros(L)
            for j in range(L):
                pri_val = float(Ysv[:, j].mean()); a_, b_ = fit_platt(logit(ex["src_val"][:, j]), Ysv[:, j]); Pc = sig(a_ * logit(P[:, j]) + b_)
                Q_cal[:, j] = Pc                                                   # variant 2: calibrate on the source validation set only
                Q_raw[:, j], est_raw[j] = saerens(P[:, j], float(SRC[src]["Y"][SRC[src]["tr"]][:, j].mean()))      # variant 1: EM on uncalibrated posteriors
                Q_em[:, j], est_em[j] = saerens(Pc, pri_val)                       # variant 3: calibrate on source validation, then EM
            f = lambda Q: np.mean([ece_of(Y[:, j], Q[:, j]) for j in range(L)]); c = lambda Q: np.mean([Q[:, j].mean() - prev_t[j] for j in range(L)])
            rows.append(dict(arch=a, direction=dname(src, T), seed=s, ece_none=f(P), ece_em_uncal=f(Q_raw), ece_cal_only=f(Q_cal), ece_cal_em=f(Q_em),
                             citl_none=c(P), citl_cal_em=c(Q_em), prev_err_source=np.mean(np.abs(SRC[src]["Y"][SRC[src]["tr"]].mean(0) - prev_t)),
                             prev_err_em_uncal=np.mean(np.abs(est_raw - prev_t)), prev_err_cal_em=np.mean(np.abs(est_em - prev_t))))
SH = pd.DataFrame(rows)
if len(SH) == 0: print("no inputs for the label-free correction: see the G2 cell")
else:
    save_tab(SH, "table14_prior_shift_numeric", index=False)
    shs = SH.groupby(["direction", "arch"]).mean(numeric_only=True).round(4); display(shs); save_tab(shs, "table14_prior_shift_correction")
    print("ece_*: ECE after each variant. prev_err_*: mean absolute error of the assumed target prevalence (source prevalence vs the EM estimate).")
    print("AUROC is unchanged by all variants (monotone). If ece_em_uncal is WORSE than ece_none, plain EM failed because the posteriors are not calibrated.")
    dirs_ = list(dict.fromkeys(SH.direction)); w = .2; fig, ax = plt.subplots(figsize=(COL2, 3.0))
    for i, (col, nm, hat) in enumerate([("ece_none", "none", ""), ("ece_em_uncal", "EM, uncalibrated", "\\\\"), ("ece_cal_only", "source-calibrated only", "xx"), ("ece_cal_em", "source-calibrated + EM", "///")]):
        for k, a in enumerate(A3):
            g_ = SH[SH.arch == a].groupby("direction")[col].mean().reindex(dirs_)
            ax.bar(np.arange(len(dirs_)) + (i - 1.5) * w + (k - 1) * w / 3.4, g_.values, w / 3.4, color=AC[a], hatch=hat, edgecolor="k", lw=.3, label=f"{nm}" if k == 0 else None)
    ax.set_xticks(range(len(dirs_))); ax.set_xticklabels(dirs_); ax.set(ylabel="ECE on the external site"); ax.legend(frameon=False, fontsize=6); ax.grid(True, axis="y", alpha=.3)
    ax.set_title("Label-free correction (within each group: DenseNet121, ConvNeXt-T, ViT-S)", fontsize=7); fig.tight_layout(); save_fig(fig, "fig11_prior_shift_correction")

In [ ]:
rng = np.random.default_rng(21)
DGKEYS = [(a, ts, me) for a in A3 for ts in POOLED for me in METHS if (a, ts, me) in PR3]
PSd = {k: pm(k[0], k[1], UNSEEN[k[1]], k[2]) for k in DGKEYS}; BQ = {k: np.zeros(NB) for k in DGKEYS}
for r in tqdm(range(NB), desc="DG bootstrap"):
    ix = {T: resample(T, rng) for T in "ABC"}                                       # shared by every method and architecture = paired
    for (a, ts, me), P in PSd.items(): T = UNSEEN[ts]; BQ[(a, ts, me)][r] = mauc(YT[T][ix[T]], P[ix[T]], VALID[T])
PQ = {k: mauc(YT[UNSEEN[k[1]]], P, VALID[UNSEEN[k[1]]]) for k, P in PSd.items()}

rows = []
for base, methods in [(NAT, METHS[1:]), ("ERM (balanced)", METHS[2:])]:
    fam = []
    for a in A3:
        for ts in POOLED:
            if (a, ts, base) not in PSd: continue
            for me in methods:
                if (a, ts, me) not in PSd: continue
                d = BQ[(a, ts, me)] - BQ[(a, ts, base)]
                fam.append(dict(baseline=base, arch=a, train=ts, unseen=UNSEEN[ts], method=me, diff=PQ[(a, ts, me)] - PQ[(a, ts, base)], lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5), p=boot_p(d, NB)))
    fam = pd.DataFrame(fam); fam["p_holm"] = holm(fam.p.values); fam["significant"] = fam.p_holm < .05; rows.append(fam)
    print(f"baseline {base}: {len(fam)} comparisons, smallest attainable Holm p = {len(fam) / NB:.4f}")
DGS = pd.concat(rows, ignore_index=True); display(DGS.round(4)); save_tab(DGS, "table15_dg_vs_erm", index=False)
print("\nDG methods that BEAT their baseline after Holm correction:"); display(DGS[DGS.significant & (DGS["diff"] > 0)][["baseline", "arch", "train", "method", "diff", "lo", "hi", "p_holm"]].round(4))
print("DG methods that are significantly WORSE:", int((DGS.significant & (DGS["diff"] < 0)).sum()), "of", len(DGS))

rows = []
for (a, ts, me), d in PR3.items():
    if ts not in POOLED: continue
    T = UNSEEN[ts]; v = np.where(VALID[T])[0]
    for s, z in d.items():
        sa = [mauc(YT[t], z[t], VALID[t]) for t in SEEN_T[ts]]
        rows.append(dict(arch=a, train=ts, unseen=T, method=me, seed=s, auroc=mauc(YT[T], z[T], VALID[T]), auprc=mauprc(YT[T], z[T], VALID[T]),
                         ece=np.mean([ece_of(YT[T][:, j], z[T][:, j]) for j in v]), seen_mean=np.mean(sa), seen_worst=np.min(sa)))
DG3 = pd.DataFrame(rows); save_tab(DG3, "table16_dg_methods_numeric", index=False)
g = DG3.groupby(["arch", "train", "method"])
TABDG = pd.DataFrame({"Unseen AUROC": g.auroc.agg(lambda x: f"{x.mean():.3f} ± {x.std(ddof=1) if len(x) > 1 else 0:.3f}"), "Unseen AUPRC": g.auprc.agg(lambda x: f"{x.mean():.3f} ± {x.std(ddof=1) if len(x) > 1 else 0:.3f}"),
                      "Unseen ECE": g.ece.agg(lambda x: f"{x.mean():.3f} ± {x.std(ddof=1) if len(x) > 1 else 0:.3f}"), "Seen AUROC (mean)": g.seen_mean.agg(lambda x: f"{x.mean():.3f}"),
                      "Seen AUROC (worst)": g.seen_worst.agg(lambda x: f"{x.mean():.3f}"), "seeds": g.seed.nunique()})
display(TABDG.reindex(pd.MultiIndex.from_product([A3, SETS_ORDER, METHS], names=["arch", "train", "method"])).dropna(how="all")); save_tab(TABDG, "table16_dg_methods")

In [ ]:
# ---------------- DG figures ----------------
fig, axes = plt.subplots(1, 3, figsize=(COL2, 3.0), sharey=True); w = .26
for a_, ts in zip(axes, SETS_ORDER):
    for i, a in enumerate(A3):
        gg = DG3[(DG3.arch == a) & (DG3.train == ts)].groupby("method").auroc.agg(["mean", "std"]).reindex(METHS)
        a_.bar(np.arange(len(METHS)) + (i - 1) * w, gg["mean"], w, yerr=gg["std"].fillna(0), capsize=1, color=AC[a], edgecolor="k", lw=.3, label=a)
    a_.set_xticks(range(len(METHS))); a_.set_xticklabels(METHS, rotation=40, ha="right", fontsize=6); a_.set_title(f"{ts}→{UNSEEN[ts]}"); a_.set_ylim(.6, None); a_.grid(True, axis="y", alpha=.3)
axes[0].set_ylabel("Unseen-site macro AUROC"); axes[0].legend(frameon=False, fontsize=6); fig.tight_layout(); save_fig(fig, "fig12_dg_methods_auroc")

for base, nm, fig_name in [(NAT, "ERM (natural)", "fig13_dg_forest_vs_natural"), ("ERM (balanced)", "ERM (balanced), the fair baseline", "fig14_dg_forest_vs_balanced")]:
    rows_ = METHS[1:] if base == NAT else METHS[2:]
    fig, axes = plt.subplots(1, 3, figsize=(COL2, 2.9), sharex=True)
    for a_, ts in zip(axes, SETS_ORDER):
        for i, a in enumerate(A3):
            for _, r in DGS[(DGS.baseline == base) & (DGS.arch == a) & (DGS.train == ts)].iterrows():
                a_.errorbar(r["diff"], rows_.index(r.method) + (i - 1) * .25, xerr=[[r["diff"] - r.lo], [r.hi - r["diff"]]], fmt=MKR[a], color="k", mfc="k" if r.significant else "w", ms=4, capsize=1.5, lw=.7, label=a if r.method == rows_[0] else None)
        a_.axvline(0, color="k", ls="--", lw=.6); a_.set_yticks(range(len(rows_))); a_.set_yticklabels(rows_ if a_ is axes[0] else [""] * len(rows_)); a_.invert_yaxis()
        a_.set(title=f"{ts}→{UNSEEN[ts]}", xlabel=f"AUROC − {nm.split(',')[0]}"); a_.grid(True, axis="x", alpha=.3)
    axes[0].legend(frameon=False, fontsize=6); fig.suptitle("Filled marker = significant after Holm correction", fontsize=7); fig.tight_layout(); save_fig(fig, fig_name)

MKS = {"A+B": "o", "A+C": "s", "B+C": "^"}                                          # seen vs unseen trade-off
fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.9), sharex=False, sharey=False)
for a_, a in zip(ax, A3):
    for ts in SETS_ORDER:
        for me in METHS:
            d = DG3[(DG3.arch == a) & (DG3.train == ts) & (DG3.method == me)]
            if d.empty: continue
            a_.errorbar(d.seen_mean.mean(), d.auroc.mean(), xerr=d.seen_mean.std(ddof=1) if len(d) > 1 else 0, yerr=d.auroc.std(ddof=1) if len(d) > 1 else 0, fmt=MKS[ts], color=MC[me], mec="k", mew=.5, ms=5, capsize=1, elinewidth=.5)
    a_.set(title=a, xlabel="AUROC on seen hospitals"); a_.grid(True, alpha=.3)
ax[0].set_ylabel("AUROC on the unseen hospital")
for me in METHS: ax[0].plot([], [], "s", color=MC[me], mec="k", label=me)
for ts in SETS_ORDER: ax[1].plot([], [], MKS[ts], color="w", mec="k", label=f"{ts}→{UNSEEN[ts]}")
ax[0].legend(frameon=False, fontsize=5.5); ax[1].legend(frameon=False, fontsize=5.5); fig.tight_layout(); save_fig(fig, "fig15_seen_vs_unseen")

In [ ]:
# ---------------- training diagnostics: per-source validation curves and GroupDRO group weights ----------------
for a in ["ViT-S", "ConvNeXt-T"]:
    fig, axes = plt.subplots(1, 3, figsize=(COL2, 2.8))
    for a_, ts in zip(axes, SETS_ORDER):
        srcs = ts.split("+")
        for me in METHS:
            f = stem(a, ts, me, 0) + "_history.csv"
            if not os.path.exists(f): continue
            h = pd.read_csv(f)
            for k, src in enumerate(srcs): a_.plot(h.epoch, h[f"val_{src}"], ls="-" if k == 0 else "--", color=MC[me] if me != "GroupDRO" else "0.5", marker="o", ms=2, lw=.9)
        a_.set(title=f"{ts} (solid = {srcs[0]}, dashed = {srcs[1]})", xlabel="Epoch"); a_.grid(True, alpha=.3)
    axes[0].set_ylabel("Source-validation AUROC")
    for me in METHS: axes[0].plot([], [], color=MC[me] if me != "GroupDRO" else "0.5", label=me)
    axes[0].legend(frameon=False, fontsize=5.5); fig.suptitle(a, fontsize=8); fig.tight_layout(); save_fig(fig, f"fig16_training_curves_{a.replace('-', '')}")

rows = []
for a in ["ViT-S", "ConvNeXt-T"]:
    for ts in SETS_ORDER:
        for s in SEEDS:
            f = stem(a, ts, "GroupDRO", s) + "_last.pt"
            if os.path.exists(f):
                q = torch.load(f, map_location="cpu", weights_only=False)["q"].numpy(); rows.append(dict(arch=a, train=ts, seed=s, **{f"q_{k}": round(float(v), 3) for k, v in zip(ts.split("+"), q)}))
QW = pd.DataFrame(rows); display(QW); save_tab(QW, "table17_groupdro_final_weights", index=False)
print("A weight that is (almost) 1 for one hospital means GroupDRO collapsed onto that hospital: the 'GroupDRO' result is then effectively single-hospital training. Report this.")

# Part I: ensembles and diversity

* **I1** seed ensemble vs a single seed.
* **I2** cross-source ensemble (average the single-hospital models) vs pooled training at **equal inference cost** (pooled x1 and pooled x2), for every architecture. Only the target C is a clean test of hospital diversity: for targets A and B the ensemble contains the weak VinBigData-only model (about 12,000 images, no Edema).
* **I3** cross-architecture ensembles (DenseNet121, ConvNeXt-T, ViT-S) and how differently the architectures rank patients.

In [ ]:
rows = []                                                                              # I1: seed ensemble gain
for a in A3:
    for ts, T in ALLC:
        d = PR3.get((a, ts, NAT), {})
        if len(d) < 2: continue
        sg = [mauc(YT[T], d[s][T], VALID[T]) for s in d]; rows.append(dict(arch=a, direction=dname(ts, T), single_mean=np.mean(sg), seed_ensemble=PTM3[(a, ts, T)], gain=PTM3[(a, ts, T)] - np.mean(sg)))
E1 = pd.DataFrame(rows); display(E1.groupby("arch")[["single_mean", "seed_ensemble", "gain"]].mean().round(4)); save_tab(E1, "table18_seed_ensemble", index=False)

TGT = {"A": (["B", "C"], "B+C"), "B": (["A", "C"], "A+C"), "C": (["A", "B"], "A+B")}   # I2: cross-source ensemble vs pooled training
def members2(a, T):
    srcs, pool = TGT[T]; need = [(a, k, NAT) for k in srcs + [pool]]
    if not all(n in PR3 for n in need): return None
    S = [s for s in SEEDS if all(s in PR3[n] for n in need)]
    if len(S) < 2: return None
    return dict(cross=[np.mean([PR3[(a, k, NAT)][s][T] for k in srcs], 0) for s in S],
                pool1=[PR3[(a, pool, NAT)][s][T] for s in S],
                pool2=[np.mean([PR3[(a, pool, NAT)][s][T], PR3[(a, pool, NAT)][S[(i + 1) % len(S)]][T]], 0) for i, s in enumerate(S)])     # two pooled models = same cost as the ensemble
MB = {(a, T): members2(a, T) for a in A3 for T in "ABC"}; MB = {k: v for k, v in MB.items() if v}
ens_auc = lambda a, T, ix, k: np.mean([mauc(YT[T][ix], p[ix], VALID[T]) for p in MB[(a, T)][k]])
VAR = ["cross", "pool1", "pool2"]; BE2 = {(a, T, k): np.zeros(NB_ENS) for (a, T) in MB for k in VAR}; rng = np.random.default_rng(41)
for r in tqdm(range(NB_ENS), desc="ensemble bootstrap"):
    ix = {T: resample(T, rng) for T in "ABC"}
    for (a, T) in MB:
        for k in VAR: BE2[(a, T, k)][r] = ens_auc(a, T, ix[T], k)
PE2 = {(a, T, k): ens_auc(a, T, np.arange(len(YT[T])), k) for (a, T) in MB for k in VAR}
rows = []
for (a, T) in MB:
    for ref_, nm in [("pool1", "pooled x1"), ("pool2", "pooled x2 (equal cost)")]:
        d = BE2[(a, T, "cross")] - BE2[(a, T, ref_)]
        rows.append(dict(arch=a, target=f"{SITE[T]} ({T})", comparison=f"cross-source ensemble − {nm}", diff=PE2[(a, T, "cross")] - PE2[(a, T, ref_)], lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5), p=boot_p(d, NB_ENS)))
E2 = pd.DataFrame(rows); E2["p_holm"] = holm(E2.p.values); E2["significant"] = E2.p_holm < .05; display(E2.round(4)); save_tab(E2, "table19_cross_source_ensemble", index=False)
fig, axes = plt.subplots(1, 3, figsize=(COL2, 2.9), sharey=True); w = .26
for a_, a in zip(axes, A3):
    for i, (k, nm, c) in enumerate([("pool1", "Pooled x1", "0.75"), ("pool2", "Pooled x2", "0.5"), ("cross", "Cross-source ensemble", "0.2")]):
        Ts = [T for T in "ABC" if (a, T) in MB]; v = np.array([PE2[(a, T, k)] for T in Ts]); cc = np.array([pci(BE2[(a, T, k)]) for T in Ts])
        a_.bar(np.arange(len(Ts)) + (i - 1) * w, v, w, yerr=[np.clip(v - cc[:, 0], 0, None), np.clip(cc[:, 1] - v, 0, None)], capsize=1.5, color=c, edgecolor="k", lw=.4, label=nm)
    a_.set_xticks(range(len(Ts))); a_.set_xticklabels([f"Test {SITE[T]}" for T in Ts], fontsize=6); a_.set_title(a); a_.set_ylim(.6, None); a_.grid(True, axis="y", alpha=.3)
axes[0].set_ylabel("Macro AUROC"); axes[0].legend(frameon=False, fontsize=6); fig.tight_layout(); save_fig(fig, "fig17_cross_source_ensemble")

In [ ]:
# ---------------- I3: cross-architecture ensembles and diversity ----------------
def ens(ts, T, names): return np.mean([pm(a, ts, T) for a in names], 0)
combos = {"DenseNet121": ["DenseNet121"], "ConvNeXt-T": ["ConvNeXt-T"], "ViT-S": ["ViT-S"], "D + C": ["DenseNet121", "ConvNeXt-T"], "D + V": ["DenseNet121", "ViT-S"], "C + V": ["ConvNeXt-T", "ViT-S"], "D + C + V": A3}
okc = [(ts, T) for ts, T in EXT if all((a, ts, NAT) in PR3 for a in A3)]
rows = [dict(direction=dname(ts, T), ensemble=k, auroc=mauc(YT[T], ens(ts, T, v), VALID[T])) for ts, T in okc for k, v in combos.items()]
EN = pd.DataFrame(rows); tab = EN.pivot(index="direction", columns="ensemble", values="auroc")[list(combos)]
tab["gain over best single"] = tab["D + C + V"] - tab[["DenseNet121", "ConvNeXt-T", "ViT-S"]].max(1); display(tab.round(4)); save_tab(tab.round(4), "table20_architecture_ensembles")

KEY = ["DenseNet121", "ConvNeXt-T", "ViT-S", "D + C + V"]; BE3 = {(dname(*c), k): np.zeros(NB_ENS) for c in okc for k in KEY}; rng = np.random.default_rng(43)
for r in tqdm(range(NB_ENS), desc="architecture-ensemble bootstrap"):
    ix = {T: resample(T, rng) for T in "ABC"}
    for ts, T in okc:
        for k in KEY: BE3[(dname(ts, T), k)][r] = mauc(YT[T][ix[T]], ens(ts, T, combos[k])[ix[T]], VALID[T])
fig, ax = plt.subplots(figsize=(COL2, 3.0)); w = .2; cols = ["0.2", "0.55", "0.85", "0.98"]
for i, k in enumerate(KEY):
    v = np.array([tab.loc[dname(*c), k] for c in okc]); cc = np.array([pci(BE3[(dname(*c), k)]) for c in okc])
    ax.bar(np.arange(len(okc)) + (i - 1.5) * w, v, w, yerr=[np.clip(v - cc[:, 0], 0, None), np.clip(cc[:, 1] - v, 0, None)], capsize=1, color=cols[i], edgecolor="k", lw=.4, hatch="///" if i == 3 else "", label=k)
ax.set_xticks(range(len(okc))); ax.set_xticklabels([dname(*c) for c in okc], rotation=40, ha="right"); ax.set_ylim(.55, None); ax.set(ylabel="Macro AUROC"); ax.legend(frameon=False, ncol=2, fontsize=6); ax.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig18_architecture_ensembles")

def rank_corr(P1, P2, T):                                                              # mean Spearman correlation of scores over valid findings
    v = np.where(VALID[T])[0]; return np.mean([np.corrcoef(rankdata(P1[:, j]), rankdata(P2[:, j]))[0, 1] for j in v])
rows = [dict(direction=dname(ts, T), pair=f"{a} vs {b}", spearman=rank_corr(pm(a, ts, T), pm(b, ts, T), T)) for ts, T in okc for a, b in combinations(A3, 2)]
DV = pd.DataFrame(rows).pivot(index="direction", columns="pair", values="spearman"); display(DV.round(3)); save_tab(DV.round(3), "table21_architecture_diversity")
fig, ax = plt.subplots(figsize=(COL1 * 1.6, 3.2)); im = ax.imshow(DV.values, cmap="viridis", vmin=DV.values.min() - .02, vmax=1)
for i in range(DV.shape[0]):
    for j in range(DV.shape[1]): ax.text(j, i, f"{DV.values[i, j]:.2f}", ha="center", va="center", color="w", fontsize=6)
ax.set_xticks(range(DV.shape[1])); ax.set_xticklabels([c.replace(" vs ", "\nvs\n") for c in DV.columns], fontsize=5.5); ax.set_yticks(range(DV.shape[0])); ax.set_yticklabels(DV.index); ax.grid(False)
fig.colorbar(im, ax=ax, label="Spearman correlation of scores"); ax.set_title("Score agreement between architectures"); fig.tight_layout(); save_fig(fig, "fig19_architecture_diversity")

# Part J: representation, site probe and Grad-CAM

* **J1** How strongly each architecture's features identify the hospital (5-fold logistic-regression probe, chance = 0.33), against a raw-pixel (28 x 28) baseline, plus t-SNE maps. If raw pixels already separate the hospitals, image format and acquisition differences are a likely source of separability.
* **J2** Grad-CAM examples (ViT-S true positives, false positives, false negatives on external sites) and a **same-image comparison of the three architectures**, with two quantitative checks: the share of Grad-CAM mass in the outer 10% border (a uniform map would put 36% there), and the agreement between architectures (Pearson r and top-20% IoU).

Grad-CAM is **descriptive**: it does not prove what a model uses. The conv CAMs are 7 x 7 and the ViT CAM is 14 x 14 (token grid), which affects the comparison, and the paper must say so.

In [ ]:
CK_AB = {"DenseNet121": os.path.join(OUT, "dg", "erm_nat_e7_seed0.pt"), "ConvNeXt-T": os.path.join(OUT, "c_source", "convnext_ApB_erm_nat_e7_seed0.pt"), "ViT-S": stem("ViT-S", "A+B", NAT, 0) + ".pt"}
AV35 = [a for a in A3 if os.path.exists(CK_AB[a])]
for a in A3:
    if a not in AV35: print(f"SKIPPED {a}: checkpoint not found -> {CK_AB[a]}   (edit CK_AB with the real path of your A+B ERM natural seed-0 checkpoint)")
@torch.inference_mode()
def emb(m, a, A, idx):
    out = []
    for i in range(0, len(idx), 100):
        with torch.autocast("cuda", enabled=(dev == "cuda")): out.append(m.forward_head(m.forward_features(gpu_batch(A, idx[i:i + 100], a=a)), pre_logits=True).float().cpu())
    return torch.cat(out).numpy()
rng = np.random.default_rng(0); SEL = {T: np.sort(rng.choice(len(TEST[T][1]), 600, replace=False)) for T in "ABC"}; site = np.repeat(list("ABC"), 600); EMB = {}
for a in AV35:
    m = load_arch(a, CK_AB[a]); EMB[a] = np.concatenate([emb(m, a, TEST[T][0], TEST[T][1][SEL[T]]) for T in "ABC"]); del m; torch.cuda.empty_cache()
raw = np.concatenate([F.adaptive_avg_pool2d(torch.from_numpy(np.ascontiguousarray(TEST[T][0][TEST[T][1][SEL[T]]])).float()[:, None], 28).flatten(1).numpy() for T in "ABC"])
probe = lambda X: cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), X, site, cv=5).mean()
PB = {**{a: probe(EMB[a]) for a in AV35}, "raw pixels (28x28)": probe(raw)}
PBt = pd.Series(PB, name="site-probe accuracy (chance 0.33)").round(3).to_frame(); display(PBt); save_tab(PBt, "table22_site_probe")
fig, axes = plt.subplots(1, len(AV35), figsize=(COL2 * len(AV35) / 3, 2.9), squeeze=False); axes = axes[0]
for a_, a in zip(axes, AV35):
    Z2 = TSNE(2, perplexity=35, random_state=0, init="pca").fit_transform(EMB[a])
    for T, c in zip("ABC", ["C0", "C1", "C2"]): a_.scatter(*Z2[site == T].T, s=3, alpha=.6, color=c, label=SITE[T])
    a_.set(title=f"{a}\nprobe acc {PB[a]:.2f}", xticks=[], yticks=[])
axes[0].legend(frameon=False, markerscale=3, fontsize=6); fig.tight_layout(); save_fig(fig, "fig20_tsne_site")

In [ ]:
class ViTCAM:                                                                          # Grad-CAM for ViT: last block norm1 tokens, patch tokens only
    def __init__(self, m):
        self.m = m; self.np_ = m.num_prefix_tokens; m.blocks[-1].norm1.register_forward_hook(self._fwd)
    def _fwd(self, mod, i, out):
        self.act = out
        if out.requires_grad: out.register_hook(lambda g: setattr(self, "grad", g))
    def __call__(self, x, j):                                                          # j: int or LongTensor [B] (finding per image)
        self.m.zero_grad(set_to_none=True)
        with torch.enable_grad():
            z = self.m(x); j = torch.full((len(x),), j, device=x.device) if isinstance(j, int) else j
            z.gather(1, j.view(-1, 1)).sum().backward()
        a_, g_ = self.act[:, self.np_:].detach(), self.grad[:, self.np_:].detach()
        cam = torch.relu((g_.mean(1, keepdim=True) * a_).sum(-1)); n = int(cam.shape[1] ** .5)
        return F.interpolate(cam.view(-1, 1, n, n), size=224, mode="bilinear", align_corners=False)[:, 0].cpu().numpy(), torch.sigmoid(z.detach()).cpu().numpy()
def make_cam(m, a):
    if a == "ViT-S": vc = ViTCAM(m); return lambda x, j: vc(x, j)[0]
    def f(x, j):                                                                       # conv Grad-CAM on the last feature map (DenseNet121, ConvNeXt-T)
        with torch.enable_grad():
            fm = m.forward_features(x); out = m.forward_head(fm); g_ = torch.autograd.grad(out[:, j].sum(), fm)[0]
        cam = torch.relu((g_.mean((2, 3), keepdim=True) * fm).sum(1, keepdim=True))
        return F.interpolate(cam, size=224, mode="bilinear", align_corners=False)[:, 0].detach().cpu().numpy()
    return f
nrm = lambda c: c / (c.reshape(len(c), -1).max(1)[:, None, None] + 1e-8)
def bp(ax, data, names):                                                               # boxplot that works on old and new Matplotlib
    try: ax.boxplot(data, tick_labels=names, showfliers=False)
    except TypeError: ax.boxplot(data, labels=names, showfliers=False)

# --- J2a: ViT-S true positives / false positives / false negatives on external sites ---
VCASES = [("A", "B", "A→B (NIH model on CheXpert)"), ("B", "A", "B→A (CheXpert model on NIH)"), ("A+B", "C", "A+B→C (pooled model on VinBigData)")]
for ts, T, ttl in VCASES:
    ck = stem("ViT-S", ts, NAT, 0) + ".pt"
    if not os.path.exists(ck): print("skipped (no checkpoint):", ck); continue
    m = load_arch("ViT-S", ck); cam = ViTCAM(m); A_, idxs = TEST[T]; P = PR3[("ViT-S", ts, NAT)][0][T]; Y = YT[T]
    for lab in ["Cardiomegaly", "Pleural Effusion"]:
        j = LABELS6.index(lab); p, y = P[:, j], Y[:, j]
        pick = {"True positive": np.where(y == 1)[0][np.argsort(-p[y == 1])[:4]], "False positive": np.where(y == 0)[0][np.argsort(-p[y == 0])[:4]], "False negative": np.where(y == 1)[0][np.argsort(p[y == 1])[:4]]}
        fig, ax = plt.subplots(3, 4, figsize=(COL2, 5.4))
        for r_, (nm, ii) in enumerate(pick.items()):
            cams, probs = cam(gpu_batch(A_, idxs[ii], a="ViT-S"), j)
            for c_ in range(4):
                a_ = ax[r_, c_]; a_.imshow(A_[idxs[ii[c_]]], cmap="gray"); a_.imshow(nrm(cams)[c_], cmap="jet", alpha=.4); a_.axis("off"); a_.set_title(f"{nm}\np={probs[c_, j]:.2f}", fontsize=6)
        fig.suptitle(f"{ttl} · {lab}", fontsize=8); fig.tight_layout(); save_fig(fig, f"fig21_gradcam_vit_{ts.replace('+', 'p')}_to_{T}_{lab.replace(' ', '')}")
    del m, cam; torch.cuda.empty_cache()

# --- J2b: same images, three architectures (NIH-trained models on CheXpert positives) ---
PA_ = {"DenseNet121": os.path.join(OUT, "nih_train", "densenet121_nih_e7_seed0.pt"), "ConvNeXt-T": os.path.join(OUT, "part34", "convnext_tiny_nih_e7_seed0.pt"), "ViT-S": stem("ViT-S", "A", NAT, 0) + ".pt"}
AV36 = [a for a in A3 if os.path.exists(PA_[a])]
for a in A3:
    if a not in AV36: print(f"SKIPPED {a}: checkpoint not found -> {PA_[a]}")
LAB = "Pleural Effusion"; jl = LABELS6.index(LAB); pos = np.where(YT["B"][:, jl] == 1)[0]; rng = np.random.default_rng(7); NST = 150; idx_st = np.sort(rng.choice(pos, NST, replace=False)); show = idx_st[:6]
CAMS = {}
for a in AV36:
    m = load_arch(a, PA_[a]); cam = make_cam(m, a); out = []
    for i in range(0, NST, 25): out.append(cam(gpu_batch(CX_T, idx_st[i:i + 25], a=a), jl))
    CAMS[a] = nrm(np.concatenate(out)); del m, cam; torch.cuda.empty_cache()
fig, ax = plt.subplots(len(AV36), 6, figsize=(COL2, 1.3 * len(AV36) + .3), squeeze=False)
for r_, a in enumerate(AV36):
    for c_ in range(6): ax[r_, c_].imshow(CX_T[show[c_]], cmap="gray"); ax[r_, c_].imshow(CAMS[a][c_], cmap="jet", alpha=.4); ax[r_, c_].axis("off")
    ax[r_, 0].set_title(a, fontsize=7, loc="left")
fig.suptitle(f"Grad-CAM, {LAB}: NIH-trained models on CheXpert positives (same images in every row)", fontsize=7); fig.tight_layout(); save_fig(fig, "fig22_gradcam_three_architectures")

def bshare(c, f=.1):
    k = int(224 * f); mk_ = np.ones((224, 224), bool); mk_[k:-k, k:-k] = False; return c[:, mk_].sum(1) / (c.reshape(len(c), -1).sum(1) + 1e-8)
down = lambda c: F.avg_pool2d(torch.from_numpy(c)[:, None], 4)[:, 0].numpy().reshape(len(c), -1)
def top_iou(c1, c2, q=80):
    a_, b_ = c1 >= np.percentile(c1, q, axis=1, keepdims=True), c2 >= np.percentile(c2, q, axis=1, keepdims=True); return (a_ & b_).sum(1) / ((a_ | b_).sum(1) + 1e-8)
BR3 = pd.DataFrame([dict(arch=a, border_share=x) for a in AV36 for x in bshare(CAMS[a])]); BRt = BR3.groupby("arch").border_share.agg(["mean", "std"]).round(3)
print("uniform CAM = 0.36 in the outer 10% frame"); display(BRt); save_tab(BRt, "table23_gradcam_border")
rows = []
for a, b in combinations(AV36, 2):
    da, db = down(CAMS[a]), down(CAMS[b]); pear = np.array([np.corrcoef(x, y)[0, 1] for x, y in zip(da, db)]); iou = top_iou(da, db)
    rows += [dict(pair=f"{a} vs {b}", metric="Pearson r", value=v) for v in pear] + [dict(pair=f"{a} vs {b}", metric="top-20% IoU", value=v) for v in iou]
AG = pd.DataFrame(rows); AGt = AG.groupby(["pair", "metric"]).value.agg(["mean", "std"]).round(3); display(AGt); save_tab(AGt, "table24_gradcam_agreement")
fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.7))
bp(ax[0], [BR3[BR3.arch == a].border_share for a in AV36], [a.replace("-", "-\n") for a in AV36]); ax[0].axhline(.36, color="r", ls="--", lw=.7); ax[0].set(ylabel="CAM mass in outer 10% border", title="Border share")
for a_, met in zip(ax[1:], ["Pearson r", "top-20% IoU"]):
    g = AG[AG.metric == met]; ps_ = list(g.pair.unique()); bp(a_, [g[g.pair == p].value for p in ps_], [p.replace(" vs ", "\nvs\n") for p in ps_]); a_.set(title=f"Agreement ({met})")
for a_ in ax: a_.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig23_gradcam_statistics")

# Part K: cost, summary, export, methods text, limitations

In [ ]:
rows = []; HP = {"DenseNet121": os.path.join(OUT, "nih_train", "densenet121_nih_e7_seed0_history.csv"), "ConvNeXt-T": os.path.join(OUT, "part34", "convnext_tiny_nih_e7_seed0_history.csv"), "ViT-S": stem("ViT-S", "A", NAT, 0) + "_history.csv"}
for a in A3:
    m = timm.create_model(ARCH_CFG[a]["timm"], pretrained=False, num_classes=L).to(dev); npar = sum(p.numel() for p in m.parameters()) / 1e6; ips = mem = np.nan
    if dev == "cuda":
        m.eval(); x = torch.randn(128, 3, 224, 224, device=dev)
        with torch.inference_mode(), torch.autocast("cuda"):
            for _ in range(5): m(x)
            torch.cuda.synchronize(); t0 = time.time()
            for _ in range(20): m(x)
            torch.cuda.synchronize(); ips = 20 * 128 / (time.time() - t0)
        m.train(); torch.cuda.reset_peak_memory_stats(); opt = torch.optim.AdamW(m.parameters(), 1e-4); xb_ = torch.randn(BS, 3, 224, 224, device=dev)
        with torch.autocast("cuda"): loss = m(xb_).float().mean()
        loss.backward(); opt.step(); torch.cuda.synchronize(); mem = torch.cuda.max_memory_allocated() / 1e9; del opt, xb_, x, loss
    try: h = pd.read_csv(HP[a]); ep_min = h["sec"].mean() / 60 if "sec" in h else np.nan
    except Exception: ep_min = np.nan
    rows.append({"architecture": a, "params (M)": round(npar, 1), "inference img/s (fp16, bs128)": round(ips) if ips == ips else "n/a", f"train step peak GB (bs{BS})": round(mem, 1) if mem == mem else "n/a",
                 "min/epoch on NIH (from logs)": round(ep_min, 1) if ep_min == ep_min else "n/a"}); del m; gc.collect(); torch.cuda.empty_cache()
EF = pd.DataFrame(rows); display(EF); save_tab(EF, "table25_efficiency", index=False)

In [ ]:
# ---------------- automatic summary of the key numbers (check every sentence of the paper against this) ----------------
S = {}
S["asymmetry"] = {a: dict(est=float(GT[(GT.arch == a) & GT.quantity.str.startswith("Asym")].estimate.iloc[0]), lo=float(GT[(GT.arch == a) & GT.quantity.str.startswith("Asym")].lo.iloc[0]),
                          hi=float(GT[(GT.arch == a) & GT.quantity.str.startswith("Asym")].hi.iloc[0])) for a in GP3}
S["asymmetry_ci_excludes_0"] = {a: not (v["lo"] <= 0 <= v["hi"]) for a, v in S["asymmetry"].items()}
S["asymmetry_differs_between_architectures"] = {r.pair: (not bool(r.CI_includes_0)) for r in DA.itertuples()}
S["n_architecture_comparisons"] = int(len(PD3)); S["n_significant_external_architecture_differences"] = int((PD3.significant & ~PD3.seen).sum())
S["dg_beats_natural"] = DGS[(DGS.baseline == NAT) & DGS.significant & (DGS["diff"] > 0)][["arch", "train", "method", "diff"]].round(4).to_dict("records")
S["dg_beats_balanced"] = DGS[(DGS.baseline == "ERM (balanced)") & DGS.significant & (DGS["diff"] > 0)][["arch", "train", "method", "diff"]].round(4).to_dict("records")
S["dg_significantly_worse_than_natural"] = int((DGS[DGS.baseline == NAT].significant & (DGS[DGS.baseline == NAT]["diff"] < 0)).sum()); S["dg_comparisons_vs_natural"] = int((DGS.baseline == NAT).sum())
if "PBt" in globals(): S["site_probe"] = PBt.iloc[:, 0].to_dict()
if "BRt" in globals(): S["gradcam_border_share_mean"] = BRt["mean"].to_dict()
if "rect" in globals(): S["recalibration_recovery_percent"] = {str(k): v for k, v in rect.to_dict("index").items()}
json.dump(S, open(os.path.join(MASTER, "RESULTS_SUMMARY.json"), "w"), indent=2, default=str)
for k, v in S.items(): print(f"{k}: {v}")

In [ ]:
shutil.make_archive(os.path.join(OUT, "MASTER_bundle"), "zip", MASTER)
print("bundle:", os.path.join(OUT, "MASTER_bundle.zip"))
print("figures:", sorted(f for f in os.listdir(FIG) if f.endswith(".png")))
print("tables :", sorted(f for f in os.listdir(TABD) if f.endswith(".csv")))

## Figure and table index (what each one supports)

| Item | Claim it supports | Paper section |
|---|---|---|
| Table 1, 2, 3, Fig. 1 | the sites differ in size, prevalence and view mix | Data |
| Table 4, Fig. 2 | full transfer matrix for three architectures | Results |
| Table 5, 6, 7, Fig. 3 | the NIH→CheXpert gap is much larger than CheXpert→NIH, in every architecture, with paired CIs | Results (main claim) |
| Table 8, Fig. 4, 5, 6 | which architecture differences are real (Holm corrected) and where | Results |
| Table 9, 10, Fig. 7 | view position (case mix) and the gap | Analysis |
| Table 11, Fig. 8, 9 | calibration under shift | Calibration |
| Table 12, Fig. 10 | a label-free prior-shift correction | Calibration |
| Table 13, 14, Fig. 11 | how many target labels repair calibration and F1 (and that AUROC does not change) | Calibration |
| Table 15, 16, 17, Fig. 12-16 | DG methods vs ERM, with the fair (balanced) baseline and the GroupDRO weights | Results (DG) |
| Table 18-21, Fig. 17-19 | ensembling and architecture diversity | Analysis |
| Table 22, Fig. 20 | features identify the hospital, raw pixels too | Representation |
| Table 23, 24, Fig. 21-23 | Grad-CAM examples, border share, agreement (descriptive only) | Explainability |
| Table 25 | cost of each architecture | Appendix |

## Methods text (edit the bracketed parts)

*Data.* We used NIH ChestX-ray14 (A; official test list, patient-level train/validation split), CheXpert (B; patient-level split, uncertain labels treated as [your policy]) and VinBigData, built from VinDr-CXR (C; 15,000 images, a finding positive when at least 2 of 3 radiologists marked it). Six findings were studied (Cardiomegaly, Edema, Consolidation, Atelectasis, Pneumothorax, Pleural Effusion); C has no Edema, so every metric involving C uses the other five and its loss masks Edema. Images were resized to 224 x 224.

*Models.* DenseNet121, ConvNeXt-T and ViT-S (ImageNet pretraining) were fine-tuned with binary cross-entropy for 7 epochs, three seeds, each with its standard recipe (Part B). Models were trained on A, B, C, A+B, A+C and B+C. The checkpoint with the highest mean source-validation AUROC was kept; no target data were used for selection. DG methods (DANN, weight 0.1; CORAL, weight 1.0; GroupDRO, eta 0.01) and ERM with balanced sampling were run on the pooled sets.

*Statistics.* Macro AUROC over findings present in the test site. Confidence intervals and p-values use a paired patient-level bootstrap with [NB] resamples shared by all models (image-level for C); multiple comparisons were Holm-corrected within each family. The gap is in-hospital minus external AUROC; asymmetry is the NIH→CheXpert gap minus the CheXpert→NIH gap.

## Limitations to state in the paper

* Only two hospital sources with patient identifiers plus one easier site; no MIMIC-CXR or PadChest.
* VinBigData has no Edema and no patient identifier; cross-site comparisons that involve C use five findings and an image-level bootstrap.
* Images are 224 x 224; behaviour at higher resolution was not tested.
* Each architecture used its standard recipe; the three recipes are not identical, and no per-architecture hyperparameter search was run.
* DG methods use default settings. GroupDRO's raw-loss hardness can collapse onto one hospital (Table 17). The equal-budget tuning study (`REFER2`) is single-seed and its CORAL runs did not train (validation AUROC about 0.5 in every trial), so it must not be cited as evidence about CORAL.
* DenseNet121 GroupDRO on the three pooled sets with VinBigData predictions is not available.
* Grad-CAM is descriptive. The label-free prior-shift correction assumes calibrated source posteriors.
* CheXpert labels come from an automatic labeler and NIH labels from report mining, so label noise differs by site.